# 🎬 4DGaussians-Enhanced: Google Colab Workflow

Bu notebook, 4D Gaussian Splatting modellerini Google Colab'de eğitmek için eksiksiz bir iş akışı sağlar.

## Özellikler
- ✅ SAM2.1 + YOLO ile otomatik maske oluşturma
- ✅ COLMAP desteği (ham resimlerden kamera poz hesaplama)
- ✅ Maske önizleme ve doğrulama
- ✅ Maske-ağırlıklı loss ile eğitim
- ✅ Eğitim ön ayarları (quick_test, standard, high_quality, fast_motion)
- ✅ Video render ve PLY export

## Donanım Gereksinimleri
- Önerilen: A100 (Colab Pro)
- Minimum: T4 (Ücretsiz) - quick_test preset kullanın

---

## 📦 Cell 1: Kurulum (Installation)

Tüm bağımlılıkları kurar: 4DGaussians, SAM2, COLMAP, C++ submodule'ler

In [ ]:
# ============================================================
# CELL 1: INSTALLATION (getcwd HATASI DÜZELTİLDİ)
# ============================================================
import os
import sys
import shutil

PROJECT_DIR = "/content/4DGaussians-Enhanced"

# ==========================================
# 🚨 KRİTİK DÜZELTME: GÜVENLİ BÖLGEYE ÇIK
# ==========================================
# Eğer zaten projenin içindeysek, silmeden önce dışarı çıkmalıyız.
# Yoksa "getcwd: cannot access parent directories" hatası alırız.
os.chdir("/content")
print(f"📍 Güvenli ana dizine geçildi: {os.getcwd()}")

# 1. Temizlik (Temiz bir başlangıç için)
if os.path.exists(PROJECT_DIR):
    print(f"🧹 Eski kurulum temizleniyor: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

# 2. Repo'yu Klonla
print("\n📥 Repo klonlanıyor (Loglar açık)...")
!git clone https://github.com/semhfe/4DGaussians-Enhanced.git

# 3. Proje Klasörüne Gir
os.chdir(PROJECT_DIR)
print(f"📂 Proje dizinine girildi: {os.getcwd()}")

# 4. Doğru Branch'e Geç
print("\n🔀 'copilot/fix-4dgaussians-enhanced-errors' branch'ine geçiliyor...")
!git checkout copilot/fix-4dgaussians-enhanced-errors

# 5. Alt Modülleri İndir (KRİTİK ADIM)
print("\n📦 Alt modüller (Submodules) indiriliyor...")
!git submodule update --init --recursive

# 6. requirements.txt Düzenleme
print("\n🔧 'requirements.txt' düzenleniyor (Torch/MMCV temizliği)...")
!sed -i '/torch/d' requirements.txt
!sed -i '/mmcv/d' requirements.txt
!cat requirements.txt | head -n 5

# 7. Bağımlılıkları Yükleme (LOGLAR AÇIK)
print("\n📦 Python kütüphaneleri kuruluyor (Detaylı çıktı)...")
!pip install matplotlib lpips plyfile pytorch_msssim open3d imageio[ffmpeg] opencv-python
!pip install ultralytics supervision huggingface_hub
# SAM2'yi kaynaktan kur
!pip install "git+https://github.com/facebookresearch/sam2.git"

# 8. Setup Scriptini Çalıştırma
print("\n🔧 Setup scripti çalıştırılıyor (C++ Yamaları, Ninja & COLMAP)...")
if os.path.exists("scripts/colab_setup.py"):
    !python scripts/colab_setup.py
else:
    print("❌ HATA: 'scripts/colab_setup.py' dosyası bulunamadı!")
    print("   Lütfen branch isminin doğru olduğundan emin olun.")
    # Dosya yapısını kontrol et
    if os.path.exists("scripts"):
        print(f"   Mevcut dosyalar: {os.listdir('scripts')}")
    else:
        print("   'scripts' klasörü bile yok! Klonlama hatalı olabilir.")

print("\n" + "="*50)
print("✅ Kurulum tamamlandı! (Lütfen yukarıdaki loglarda 'error' olup olmadığını kontrol edin)")
print("="*50)

📍 Güvenli ana dizine geçildi: /content

📥 Repo klonlanıyor (Loglar açık)...
Cloning into '4DGaussians-Enhanced'...
remote: Enumerating objects: 2706, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 2706 (delta 39), reused 50 (delta 21), pack-reused 2625 (from 1)
Receiving objects: 100% (2706/2706), 66.49 MiB | 48.08 MiB/s, done.
Resolving deltas: 100% (1255/1255), done.
📂 Proje dizinine girildi: /content/4DGaussians-Enhanced

🔀 'copilot/fix-4dgaussians-enhanced-errors' branch'ine geçiliyor...
Branch 'copilot/fix-4dgaussians-enhanced-errors' set up to track remote branch 'copilot/fix-4dgaussians-enhanced-errors' from 'origin'.
Switched to a new branch 'copilot/fix-4dgaussians-enhanced-errors'

📦 Alt modüller (Submodules) indiriliyor...
Submodule 'submodules/depth-diff-gaussian-rasterization' (https://github.com/ingra14m/depth-diff-gaussian-rasterization) registered for path 'submodules/depth-diff-gaussian-rasterization'


In [ ]:
# ============================================================
# CELL 1.5: FINAL COMPILATION (C++ Modüllerini Derle)
# ============================================================
import os
import sys

# Proje dizininde olduğumuzdan emin olalım
os.chdir("/content/4DGaussians-Enhanced")

print("🚀 Rasterizer ve Simple-KNN derleniyor (Bu işlem 2-3 dk sürebilir)...")

# 1. Rasterizer Derleme
print("\n📦 Compiling Diff-Gaussian-Rasterization...")
!pip install -e submodules/depth-diff-gaussian-rasterization

# 2. KNN Derleme
print("\n📦 Compiling Simple-KNN...")
!pip install -e submodules/simple-knn

print("\n✅ Derleme tamamlandı! Artık Cell 2'ye geçebilirsiniz.")

🚀 Rasterizer ve Simple-KNN derleniyor (Bu işlem 2-3 dk sürebilir)...

📦 Compiling Diff-Gaussian-Rasterization...
Obtaining file:///content/4DGaussians-Enhanced/submodules/depth-diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
  DEPRECATION: Legacy editable install of diff-gaussian-rasterization==0.0.0 from file:///content/4DGaussians-Enhanced/submodules/depth-diff-gaussian-rasterization (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for diff-gaussian-rasterization

📦 Compiling Simple-KNN...
Obtaining file:///content/4DGaussians-Enhanced/submodules/simple-knn
  Preparing m

## 📁 Cell 2: Veri Hazırlama (Data Setup)

Google Drive'ı mount eder, veriyi unzip eder ve formatı doğrular.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
import os
import zipfile
import shutil

print("="*60)
print("📁 Veri Hazırlama")
print("="*60)

# Step 1: Mount Google Drive
print("\n📂 Google Drive mount ediliyor...")
drive.mount('/content/drive')
print("✅ Drive mount edildi")

# Step 2: Configure paths
# BURADAN DÜZENLEYIN: Veri yollarınızı belirtin
DATA_SOURCE = "/content/drive/MyDrive/4DGS_project/input/added_environment.zip"  # Zip dosyası veya klasör yolu
OUTPUT_BASE = "/content/drive/MyDrive/4DGS_project/output"  # Çıktıların kaydedileceği Drive klasörü

# Local processing paths (faster than Drive)
LOCAL_DATA = "/content/data/my_scene"  # Lokal veri klasörü (işleme için)
LOCAL_OUTPUT = "/content/output"  # Lokal çıktı (eğitim için)

# Step 3: Extract or copy data to local disk
os.makedirs(LOCAL_DATA, exist_ok=True)

if DATA_SOURCE.endswith('.zip'):
    if not os.path.exists(DATA_SOURCE):
        print(f"\n❌ Hata: Zip dosyası bulunamadı: {DATA_SOURCE}")
        print("   Lütfen DATA_SOURCE değişkenini güncelleyin")
    else:
        print(f"\n📦 Zip açılıyor: {DATA_SOURCE}")
        with zipfile.ZipFile(DATA_SOURCE, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_DATA)
        print(f"✅ Zip açıldı: {LOCAL_DATA}")
else:
    if not os.path.exists(DATA_SOURCE):
        print(f"\n❌ Hata: Klasör bulunamadı: {DATA_SOURCE}")
        print("   Lütfen DATA_SOURCE değişkenini güncelleyin")
    else:
        print(f"\n📂 Veri kopyalanıyor: {DATA_SOURCE} -> {LOCAL_DATA}")
        if os.path.exists(LOCAL_DATA):
            shutil.rmtree(LOCAL_DATA)
        shutil.copytree(DATA_SOURCE, LOCAL_DATA)
        print(f"✅ Veri kopyalandı")

# Step 4: Detect data format
print("\n🔍 Veri formatı algılanıyor...")
contents = os.listdir(LOCAL_DATA)
print(f"   İçerik: {contents}")

data_format = None
if 'transforms_train.json' in contents:
    data_format = 'blender'
    print("✅ Format: Blender/NeRF Synthetic")
elif 'sparse' in contents or 'images' in contents:
    data_format = 'colmap'
    print("✅ Format: COLMAP")
elif any('cam' in item for item in contents):
    data_format = 'multicam'
    print("✅ Format: Multi-camera (cam01, cam02, ...)")
elif len([f for f in contents if f.endswith(('.jpg', '.png'))]) > 0:
    data_format = 'raw_images'
    print("✅ Format: Ham resimler (COLMAP gerekli)")
else:
    print("⚠️  Format belirlenemedi. Klasör yapısını kontrol edin.")

# Step 5: Create output directory
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
os.makedirs(OUTPUT_BASE, exist_ok=True)

print("\n" + "="*60)
print("✅ Veri hazırlama tamamlandı!")
print("="*60)
print(f"\n📁 Lokal veri: {LOCAL_DATA}")
print(f"📁 Lokal çıktı: {LOCAL_OUTPUT}")
print(f"📁 Drive çıktı: {OUTPUT_BASE}")
print(f"\n📊 Format: {data_format}")

if data_format == 'raw_images':
    print("\n⚠️  Ham resimler tespit edildi!")
    print("   Cell 3'ü çalıştırarak COLMAP ile kamera pozlarını hesaplayın")
else:
    print("\n📝 Sonraki adım: Cell 4'ü çalıştırarak maske oluşturun")

📁 Veri Hazırlama

📂 Google Drive mount ediliyor...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mount edildi

📦 Zip açılıyor: /content/drive/MyDrive/4DGS_project/input/added_environment.zip
✅ Zip açıldı: /content/data/my_scene

🔍 Veri formatı algılanıyor...
   İçerik: ['added_environment', '__MACOSX']
⚠️  Format belirlenemedi. Klasör yapısını kontrol edin.

✅ Veri hazırlama tamamlandı!

📁 Lokal veri: /content/data/my_scene
📁 Lokal çıktı: /content/output
📁 Drive çıktı: /content/drive/MyDrive/4DGS_project/output

📊 Format: None

📝 Sonraki adım: Cell 4'ü çalıştırarak maske oluşturun


In [ ]:
import os
import shutil

# Assuming LOCAL_DATA is defined in a previous cell
if 'LOCAL_DATA' in locals() or 'LOCAL_DATA' in globals():
    macosx_path = os.path.join(LOCAL_DATA, '__MACOSX')
    if os.path.exists(macosx_path):
        print(f"🧹 '__MACOSX' klasörü siliniyor: {macosx_path}")
        shutil.rmtree(macosx_path)
        print("✅ '__MACOSX' klasörü silindi")
    else:
        print("ℹ️ '__MACOSX' klasörü bulunamadı, silinecek bir şey yok.")
else:
    print("❌ Hata: 'LOCAL_DATA' değişkeni tanımlı değil. Önce 'Cell 2: Veri Hazırlama' kısmını çalıştırın.")

🧹 '__MACOSX' klasörü siliniyor: /content/data/my_scene/__MACOSX
✅ '__MACOSX' klasörü silindi


## 🎯 Cell 3: COLMAP İşleme (Opsiyonel)(3.5 için burayı geç)

**Sadece ham resimleriniz varsa çalıştırın!**

COLMAP ile kamera pozlarını ve sparse point cloud'u hesaplar.

In [ ]:
# ============================================================
# CELL 2.5: COLMAP INSTALLATION (ÖN HAZIRLIK)
# ============================================================
# Bu hücreyi Cell 3'ten ÖNCE çalıştırın.
# Sistemde COLMAP yüklü değilse otomatik olarak kurar.
# ============================================================

import shutil
import os

print("🔍 COLMAP kurulumu kontrol ediliyor...")

# COLMAP komutu sistemde var mı bak
if not shutil.which("colmap"):
    print("📦 COLMAP bulunamadı. Kurulum başlatılıyor (1-2 dakika sürebilir)...")
    try:
        # 1. Paket listesini güncelle (Sessiz mod)
        !apt-get update
        # 2. COLMAP'i kur (Sessiz mod, onay istemeden)
        !apt-get install -y colmap
        print("✅ COLMAP başarıyla kuruldu!")
    except Exception as e:
        print(f"❌ Kurulum sırasında hata oluştu: {e}")
        print("👉 İpucu: '!apt-get install -y colmap' komutunu manuel deneyebilirsiniz.")
else:
    print("✅ COLMAP zaten sistemde yüklü, kuruluma gerek yok.")

# Kurulumu doğrula
print("-" * 30)
print("Sürüm Kontrolü:")
!colmap help | head -n 1

🔍 COLMAP kurulumu kontrol ediliyor...
📦 COLMAP bulunamadı. Kurulum başlatılıyor (1-2 dakika sürebilir)...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illi

In [ ]:
import os
from PIL import Image
from tqdm.notebook import tqdm

# Görsellerinin şu an bulunduğu asıl konum (Ekran görüntüne göre)
source_root = "/content/data/my_scene/added_environment"

print(f"📂 İşlem başlıyor: {source_root}")

# Klasörleri gez
count = 0
for root, dirs, files in os.walk(source_root):
    # Sadece içinde görsel olan camXX klasörlerini bul
    images = [f for f in files if f.endswith(".png")]

    if images:
        print(f"   -> {os.path.basename(root)} klasöründe {len(images)} görsel dönüştürülüyor...")

        for file in tqdm(images, leave=False):
            png_path = os.path.join(root, file)
            jpg_path = os.path.join(root, file.replace(".png", ".jpg"))

            # Eğer jpg zaten yoksa dönüştür
            if not os.path.exists(jpg_path):
                try:
                    img = Image.open(png_path)
                    # PNG'de şeffaflık (RGBA) olabilir, JPG için RGB'ye çeviriyoruz
                    rgb_img = img.convert('RGB')
                    rgb_img.save(jpg_path, quality=95)
                    count += 1
                except Exception as e:
                    print(f"Hata: {file} dönüştürülemedi. {e}")

print(f"✅ Toplam {count} görsel başarıyla JPG formatına dönüştürüldü.")
print("ℹ️ Not: Script'i çalıştırırken kaynak klasörünü (source path) şu şekilde güncellemeyi unutma:")
print(f"    {source_root}")

📂 İşlem başlıyor: /content/data/my_scene/added_environment
   -> cam06 klasöründe 66 görsel dönüştürülüyor...


  0%|          | 0/66 [00:00<?, ?it/s]

   -> cam05 klasöründe 66 görsel dönüştürülüyor...


  0%|          | 0/66 [00:00<?, ?it/s]

   -> cam08 klasöründe 66 görsel dönüştürülüyor...


...

   -> cam07 klasöründe 66 görsel dönüştürülüyor...


  0%|          | 0/66 [00:00<?, ?it/s]

   -> cam04 klasöründe 66 görsel dönüştürülüyor...


  0%|          | 0/66 [00:00<?, ?it/s]

✅ Toplam 528 görsel başarıyla JPG formatına dönüştürüldü.
ℹ️ Not: Script'i çalıştırırken kaynak klasörünü (source path) şu şekilde güncellemeyi unutma:
    /content/data/my_scene/added_environment


In [ ]:
import os

# Temizlenecek ana klasör
target_folder = "/content/data/my_scene/added_environment"

deleted_count = 0

for root, dirs, files in os.walk(target_folder):
    for file in files:
        if file.endswith(".png"):
            file_path = os.path.join(root, file)
            os.remove(file_path)
            deleted_count += 1

print(f"🧹 Temizlik tamamlandı: {deleted_count} adet PNG dosyası silindi.")

🧹 Temizlik tamamlandı: 528 adet PNG dosyası silindi.


asıl olması gereken hem jpg çeviren hem pngleri silen script

In [ ]:
import os
from PIL import Image
from tqdm.notebook import tqdm

# Görsellerin bulunduğu kök klasör
source_root = "/content/data/my_scene/added_environment"

print(f"🔄 Dönüşüm ve Temizlik Başlıyor: {source_root}")

converted_count = 0

for root, dirs, files in os.walk(source_root):
    # Sadece png dosyalarını listele
    images = [f for f in files if f.endswith(".png")]

    if images:
        print(f"   -> {os.path.basename(root)} içinde işlem yapılıyor...")

        for file in tqdm(images, leave=False):
            png_path = os.path.join(root, file)
            jpg_path = os.path.join(root, file.replace(".png", ".jpg"))

            try:
                # 1. Resmi aç ve dönüştür
                img = Image.open(png_path)
                rgb_img = img.convert('RGB')

                # 2. JPG olarak kaydet
                rgb_img.save(jpg_path, quality=95)

                # 3. Kayıt başarılıysa PNG'yi sil (Geri dönüşü yok!)
                if os.path.exists(jpg_path):
                    os.remove(png_path)
                    converted_count += 1

            except Exception as e:
                print(f"❌ Hata: {file} işlenemedi. {e}")

print(f"✅ İşlem tamamlandı. {converted_count} görsel JPG'e çevrildi ve PNG orijinalleri silindi.")

In [ ]:
import os
import sys

# --- AYARLAR ---
# Daha önce oluşturduğumuz birleştirilmiş klasör
images_path = "/content/data/my_scene/combined_first_frames"
# Çıktı klasörü
project_path = "/content/data/my_scene/colmap_output"
database_path = os.path.join(project_path, "colmap/database.db")
sparse_output_path = os.path.join(project_path, "sparse")

print("="*60)
print("🛠️ COLMAP Manuel Pipeline (CPU Modu - Güvenli)")
print("="*60)

# 1. Klasörleri Oluştur
os.makedirs(os.path.dirname(database_path), exist_ok=True)
os.makedirs(sparse_output_path, exist_ok=True)

# Veritabanı varsa sil (Temiz başlangıç için)
if os.path.exists(database_path):
    os.remove(database_path)

# --- ADIM 1: Feature Extractor (KRİTİK DÜZELTME BURADA: use_gpu=0) ---
print("\n1️⃣  Özellik Çıkarılıyor (CPU)...")
!colmap feature_extractor \
    --database_path {database_path} \
    --image_path {images_path} \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model OPENCV \
    --SiftExtraction.use_gpu 0

# --- ADIM 2: Matcher (Exhaustive - Az görsel için en iyisi) ---
print("\n2️⃣  Görseller Eşleştiriliyor...")
!colmap exhaustive_matcher \
    --database_path {database_path} \
    --SiftMatching.use_gpu 0

# --- ADIM 3: Mapper (Sparse Model Oluşturma) ---
print("\n3️⃣  3D Nokta Bulutu Oluşturuluyor (Mapper)...")
!colmap mapper \
    --database_path {database_path} \
    --image_path {images_path} \
    --output_path {sparse_output_path}

# --- ADIM 4: Model Dönüştürücü (BIN -> TXT) ---
# 4DGaussians genelde text formatını okur, garanti olsun diye çeviriyoruz.
print("\n4️⃣  Model Dönüştürülüyor...")
# Mapper bazen "0" isimli bir klasör oluşturur. Onu kontrol edelim.
model_path = os.path.join(sparse_output_path, "0")
if not os.path.exists(model_path):
    # Eğer mapper klasör oluşturmadıysa işlem başarısız olmuş olabilir
    print("⚠️ UYARI: Sparse model oluşturulamadı. Görsellerde yeterli örtüşme olmayabilir.")
else:
    !colmap model_converter \
        --input_path {model_path} \
        --output_path {model_path} \
        --output_type TXT
    print(f"\n✅ İŞLEM BAŞARIYLA TAMAMLANDI!")
    print(f"📂 Çıktılar şurada: {model_path}")
    print("📝 Artık 4DGaussians eğitimine (Cell 4 ve sonrası) geçebilirsin.")

🛠️ COLMAP Manuel Pipeline (CPU Modu - Güvenli)

1️⃣  Özellik Çıkarılıyor (CPU)...

Feature extraction

Processed file [1/8]
  Name:            cam05_frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - OPENCV
  Focal Length:    2304.00px
  Features:        1350
Processed file [2/8]
  Name:            cam01_frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - OPENCV
  Focal Length:    2304.00px
  Features:        1319
Processed file [3/8]
  Name:            cam07_frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - OPENCV
  Focal Length:    2304.00px
  Features:        1430
Processed file [4/8]
  Name:            cam02_frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - OPENCV
  Focal Length:    2304.00px
  Features:        1412
Processed file [5/8]
  Name:            cam03_frame_00001.jpg
  Dimensions:      1920 x 1080
  Camera:          #1 - OPENCV
  Focal Length:    2304.00px
  Features:        1422
Processed fi

# **CELL 3.5 colmap için extraview stratejisi**

In [ ]:
# 1. CondaColab'ı kur (Kernel Restart Gerektirir)
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:06
🔁 Restarting kernel...


In [ ]:
import os
import sys
import shutil

# ==============================================================================
# 🛠️ KURULUM VE AYARLAR
# ==============================================================================

# 1. CUDA Destekli COLMAP'i Conda üzerinden kuruyoruz
print("⚙️  CUDA Destekli COLMAP Kuruluyor (Bu işlem 1-2 dk sürebilir)...")
!conda install -c conda-forge colmap=3.8 cccl -y
print("✅ Kurulum tamamlandı. GPU Testi yapılıyor...")

# 2. Ortam Değişkenleri (A100 optimizasyonu)
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['CUDA_VISIBLE_DEVICES'] = '0' # İlk GPU'yu zorla

# ==============================================================================
# 🎛️ DENSE RECONSTRUCTION AYARLARI (BURAYI DÜZENLEYİN)
# ==============================================================================
# Dense Cloud kalitesini artırmak veya yoğunluğunu değiştirmek için bu ayarları kullanın.

# --- Patch Match Stereo (Derinlik Tahmini) ---
# window_radius: Daha büyük değerler (7-9) dokusuz alanları doldurur ama detay kaybedebilir. (Default: 5)
# num_iterations: Daha yüksek (10-15) kaliteyi artırır ama yavaştır. (Default: 5)
PM_WINDOW_RADIUS = 5
PM_NUM_ITERATIONS = 5
PM_GEOM_CONSISTENCY = 1  # 1 = Kalite artar (Multi-view tutarlılığı zorlar)

# --- Stereo Fusion (Nokta Bulutu Birleştirme) ---
# min_num_pixels: Bir noktanın oluşması için kaç kamerada görülmesi gerektiği.
# Düşük (3) = Daha çok nokta (gürültülü olabilir). Yüksek (5-7) = Daha az ama temiz nokta. (Default: 5)
SF_MIN_NUM_PIXELS = 5

# max_reproj_error: İzin verilen piksel hatası.
# Yüksek (4-5) = Daha çok nokta. Düşük (2) = Daha hassas. (Default: 2)
SF_MAX_REPROJ_ERROR = 2

# max_depth_error: Derinlik tutarlılık hatası.
# Yüksek (0.1) = Daha çok nokta. Düşük (0.01) = Daha hassas. (Default: 0.01)
SF_MAX_DEPTH_ERROR = 0.01

# ==============================================================================
# 🚀 FULL GPU PIPELINE (Sparse + Dense)
# ==============================================================================

# Kaynak (Tüm görseller: imageXX + extraXX)
COLMAP_RAW_INPUT_DIR = "/content/drive/MyDrive/4DGS_project/input/work_allviews/images"

# Geçici Çalışma Alanı
COLMAP_WORK_DIR = "/content/colmap_gpu_workspace"
COLMAP_DB = os.path.join(COLMAP_WORK_DIR, "database.db")
COLMAP_SPARSE = os.path.join(COLMAP_WORK_DIR, "sparse")
COLMAP_DENSE = os.path.join(COLMAP_WORK_DIR, "dense")

print("\n" + "="*60)
print("🚀 COLMAP A100 GPU MODU BAŞLIYOR")
print(f"📂 Kaynak: {COLMAP_RAW_INPUT_DIR}")
print("="*60)

# Temiz başlangıç
if os.path.exists(COLMAP_WORK_DIR):
    shutil.rmtree(COLMAP_WORK_DIR)
os.makedirs(COLMAP_WORK_DIR, exist_ok=True)
os.makedirs(COLMAP_SPARSE, exist_ok=True)
os.makedirs(COLMAP_DENSE, exist_ok=True)

# 1. Feature Extraction (GPU)
print("\n1️⃣  Feature Extraction (GPU)...")
!colmap feature_extractor \
    --database_path {COLMAP_DB} \
    --image_path {COLMAP_RAW_INPUT_DIR} \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model OPENCV \
    --SiftExtraction.use_gpu 1

# 2. Matcher (GPU)
print("\n2️⃣  Matcher (GPU)...")
!colmap exhaustive_matcher \
    --database_path {COLMAP_DB} \
    --SiftMatching.use_gpu 1

# 3. Mapper (Sparse Model)
print("\n3️⃣  Mapper (Sparse)...")
!colmap mapper \
    --database_path {COLMAP_DB} \
    --image_path {COLMAP_RAW_INPUT_DIR} \
    --output_path {COLMAP_SPARSE}

# Model yolunu bul (Mapper bazen 0 klasörü açar)
sparse_model_path = os.path.join(COLMAP_SPARSE, "0")
if not os.path.exists(sparse_model_path):
    sparse_model_path = COLMAP_SPARSE

# 4. Image Undistorter (Dense Hazırlığı)
print("\n4️⃣  Image Undistorter (Dense Hazırlığı)...")
!colmap image_undistorter \
    --image_path {COLMAP_RAW_INPUT_DIR} \
    --input_path {sparse_model_path} \
    --output_path {COLMAP_DENSE} \
    --output_type COLMAP \
    --max_image_size 2000

# 5. Patch Match Stereo (Depth Maps - GPU A100 GÜCÜ BURADA LAZIM)
print(f"\n5️⃣  Patch Match Stereo (Derinlik Haritaları - GPU)...")
print(f"   ⚙️ Params: Window={PM_WINDOW_RADIUS}, Iters={PM_NUM_ITERATIONS}, GeomCheck={PM_GEOM_CONSISTENCY}")

!colmap patch_match_stereo \
    --workspace_path {COLMAP_DENSE} \
    --workspace_format COLMAP \
    --PatchMatchStereo.geom_consistency {PM_GEOM_CONSISTENCY} \
    --PatchMatchStereo.window_radius {PM_WINDOW_RADIUS} \
    --PatchMatchStereo.num_iterations {PM_NUM_ITERATIONS} \
    --PatchMatchStereo.gpu_index 0

# 6. Stereo Fusion (Dense Point Cloud Üretimi)
print(f"\n6️⃣  Stereo Fusion (Dense Cloud - GPU)...")
print(f"   ⚙️ Params: MinPixels={SF_MIN_NUM_PIXELS}, MaxReproj={SF_MAX_REPROJ_ERROR}, MaxDepthErr={SF_MAX_DEPTH_ERROR}")

!colmap stereo_fusion \
    --workspace_path {COLMAP_DENSE} \
    --workspace_format COLMAP \
    --input_type geometric \
    --output_path {COLMAP_DENSE}/fused.ply \
    --StereoFusion.min_num_pixels {SF_MIN_NUM_PIXELS} \
    --StereoFusion.max_reproj_error {SF_MAX_REPROJ_ERROR} \
    --StereoFusion.max_depth_error {SF_MAX_DEPTH_ERROR}

# 7. Model Converter (TXT) - Filtreleme scripti için hazırlık
print("\n7️⃣  Sparse Model TXT'ye çevriliyor (Filtreleme için)...ध्यात्म")
!colmap model_converter \
    --input_path {sparse_model_path} \
    --output_path {sparse_model_path} \
    --output_type TXT

print("\n🎉 GPU İŞLEMLERİ TAMAMLANDI!")
print(f"☁️  Dense Point Cloud: {COLMAP_DENSE}/fused.ply")
print(f"📄 Sparse Model: {sparse_model_path}")
print("👉 Şimdi 'Adım 2: Filtreleme' kodunu çalıştırabilirsin.")

Streaming output truncated to the last 5000 lines.
 Sweep 4: 1.0592s
Iteration 1: 3.2729s
 Sweep 1: 0.5607s
 Sweep 2: 1.0586s
 Sweep 3: 0.5661s
 Sweep 4: 1.0469s
Iteration 2: 3.2325s
 Sweep 1: 0.5556s
 Sweep 2: 1.0465s
 Sweep 3: 0.5622s
 Sweep 4: 1.0348s
Iteration 3: 3.1994s
 Sweep 1: 0.5506s
 Sweep 2: 1.0365s
 Sweep 3: 0.5585s
 Sweep 4: 1.0253s
Iteration 4: 3.1710s
 Sweep 1: 0.5463s
 Sweep 2: 1.0231s
 Sweep 3: 0.5551s
 Sweep 4: 1.0137s
Iteration 5: 3.1385s
Total: 16.1401s

Writing photometric output for extra_011.png

Processing view 10 / 40 for extra_012.png

Reading inputs...

PatchMatch::Problem
-------------------
ref_image_idx: 14
src_image_idxs: 7 22 27 21 30 26 25 5 18 15 13 16 12 17 4 11 24 6 19 20

PatchMatchOptions
-----------------
max_image_size: -1
gpu_index: 0
depth_min: 2.70851
depth_max: 10.8171
window_radius: 5
window_step: 1
sigma_spatial: 5
sigma_color: 0.2
num_samples: 15
ncc_sigma: 0.6
min_triangulation_angle: 1
incident_angle_sigma: 0.9
num_iterations: 5
geom_con

In [ ]:
import os
import shutil

# ==============================================================================
# 🎯 SONUÇLARI TOPLAMA
# ==============================================================================
# GPU Workspace'den gelenler
INPUT_SPARSE_DIR = "/content/colmap_gpu_workspace/sparse/0"
INPUT_DENSE_PLY = "/content/colmap_gpu_workspace/dense/fused.ply"

# Hedef (4DGaussians Proje Klasörü)
FINAL_PROJECT_ROOT = "/content/data/my_scene/colmap_output"
FINAL_SPARSE_DIR = os.path.join(FINAL_PROJECT_ROOT, "sparse/0")

print(f"🧹 Filtreleme ve Taşıma Başlıyor...")

# Klasörleri hazırla
if os.path.exists(FINAL_SPARSE_DIR):
    shutil.rmtree(FINAL_SPARSE_DIR)
os.makedirs(FINAL_SPARSE_DIR, exist_ok=True)

# 1. Images.txt Filtreleme ("extra" silinir)
src_images = os.path.join(INPUT_SPARSE_DIR, "images.txt")
dst_images = os.path.join(FINAL_SPARSE_DIR, "images.txt")

count_kept = 0
count_deleted = 0

with open(src_images, "r") as f_in, open(dst_images, "w") as f_out:
    lines = f_in.readlines()
    # Header
    f_out.writelines([l for l in lines if l.startswith("#")])

    data_lines = [l for l in lines if not l.startswith("#")]
    i = 0
    while i < len(data_lines):
        line1 = data_lines[i]
        line2 = data_lines[i+1]
        name = line1.split()[-1]

        if "extra" in name.lower():
            count_deleted += 1
        else:
            f_out.write(line1)
            f_out.write(line2)
            count_kept += 1
        i += 2

print(f"   ✅ Sparse Model Filtrelendi: {count_kept} Ana, {count_deleted} Extra silindi.")

# 2. Diğer Dosyaları Kopyala
shutil.copy(os.path.join(INPUT_SPARSE_DIR, "cameras.txt"), FINAL_SPARSE_DIR)
shutil.copy(os.path.join(INPUT_SPARSE_DIR, "points3D.txt"), FINAL_SPARSE_DIR)

# 3. Dense PLY Kopyala (Burası kritik, artık elimizde gerçek dense var!)
final_ply_path = os.path.join(FINAL_PROJECT_ROOT, "dense_point_cloud.ply")
if os.path.exists(INPUT_DENSE_PLY):
    shutil.copy(INPUT_DENSE_PLY, final_ply_path)
    print(f"   ☁️  Dense Point Cloud (fused.ply) kopyalandı!")
else:
    print("❌ HATA: Dense PLY bulunamadı!")

# 4. BIN Dönüşümü
print("   🔄 .bin formatına dönüştürülüyor...")
!colmap model_converter --input_path {FINAL_SPARSE_DIR} --output_path {FINAL_SPARSE_DIR} --output_type BIN

print("\n🎉 HER ŞEY HAZIR! Training'e geçebilirsin.")

🧹 Filtreleme ve Taşıma Başlıyor...
   ✅ Sparse Model Filtrelendi: 8 Ana, 32 Extra silindi.
   ☁️  Dense Point Cloud (fused.ply) kopyalandı!
   🔄 .bin formatına dönüştürülüyor...

🎉 HER ŞEY HAZIR! Training'e geçebilirsin.


In [ ]:
import os
import struct

# ==============================================================================
# 🔍 KONUM AYARLARI
# ==============================================================================
TARGET_ROOT = "/content/data/my_scene/colmap_output"
SPARSE_DIR = os.path.join(TARGET_ROOT, "sparse/0")
DENSE_PLY = os.path.join(TARGET_ROOT, "dense_point_cloud.ply")

print(f"🕵️‍♂️ DOĞRULAMA BAŞLATILIYOR: {TARGET_ROOT}")
print("="*60)

# ------------------------------------------------------------------------------
# 1. IMAGES.TXT İÇERİK KONTROLÜ (Gözle Görünür Kanıt)
# ------------------------------------------------------------------------------
images_txt = os.path.join(SPARSE_DIR, "images.txt")

if not os.path.exists(images_txt):
    print("❌ KRİTİK HATA: images.txt bulunamadı! İşlem başarısız olmuş.")
else:
    print(f"📖 {images_txt} okunuyor...\n")

    with open(images_txt, "r") as f:
        lines = f.readlines()

    # Yorum satırlarını geç
    data_lines = [l for l in lines if not l.startswith("#")]

    # İstatistikler
    # COLMAP txt formatında her resim 2 satırdır (Parametreler + Noktalar)
    total_entries = len(data_lines) // 2

    print(f"   📊 Toplam Kamera Sayısı: {total_entries}")

    # İsim Kontrolü
    clean_count = 0
    dirty_count = 0
    sample_names = []

    i = 0
    while i < len(data_lines):
        line = data_lines[i]
        parts = line.strip().split()
        # Image ID, Q, T, Camera ID, NAME
        # Name en son parçadır
        img_name = parts[-1]

        if "extra" in img_name.lower():
            dirty_count += 1
            print(f"   ❌ ALARM: 'extra' dosya bulundu -> {img_name}")
        else:
            clean_count += 1
            if len(sample_names) < 5: # İlk 5 tanesini örnek al
                sample_names.append(img_name)

        i += 2

    print(f"   ✅ Temiz (Ana) Dosya Sayısı: {clean_count}")
    print(f"   🗑️  Tespit Edilen 'Extra': {dirty_count} (0 olmalı)")

    print("\n   🔎 Örnek Dosya İsimleri (İlk 5):")
    for name in sample_names:
        print(f"      -> {name}")

# ------------------------------------------------------------------------------
# 2. BIN DOSYASI KONTROLÜ (Training İçin Şart)
# ------------------------------------------------------------------------------
print("\n" + "-"*40)
expected_bins = ["images.bin", "cameras.bin", "points3D.bin"]
missing_bins = []

for b in expected_bins:
    if not os.path.exists(os.path.join(SPARSE_DIR, b)):
        missing_bins.append(b)

if missing_bins:
    print(f"❌ EKSİK BIN DOSYALARI: {missing_bins}")
    print("   Training başlamaz!")
else:
    print("✅ Tüm .bin dosyaları mevcut (images.bin, cameras.bin, points3D.bin)")
    # Basit bir byte kontrolü yapalım (Boş mu?)
    size_mb = os.path.getsize(os.path.join(SPARSE_DIR, "points3D.bin")) / (1024*1024)
    print(f"   💾 Sparse Model Boyutu (points3D.bin): {size_mb:.2f} MB")

# ------------------------------------------------------------------------------
# 3. DENSE PLY KONTROLÜ
# ------------------------------------------------------------------------------
print("\n" + "-"*40)
if os.path.exists(DENSE_PLY):
    size_mb = os.path.getsize(DENSE_PLY) / (1024*1024)
    print(f"✅ Dense Point Cloud MEVCUT: {DENSE_PLY}")
    print(f"   ⚖️  Dosya Boyutu: {size_mb:.2f} MB")

    if size_mb < 1:
        print("   ⚠️ UYARI: Dense cloud çok küçük (<1MB). İçeriği boş olabilir.")
    else:
        print("   👍 Boyut makul görünüyor.")
else:
    print("❌ HATA: dense_point_cloud.ply bulunamadı!")

print("="*60)
if dirty_count == 0 and not missing_bins and os.path.exists(DENSE_PLY):
    print("🟢 ONAYLANDI: Veri seti %100 temiz ve eğitime hazır.")
else:
    print("🔴 BAŞARISIZ: Lütfen hataları kontrol edin.")

🕵️‍♂️ DOĞRULAMA BAŞLATILIYOR: /content/data/my_scene/colmap_output
📖 /content/data/my_scene/colmap_output/sparse/0/images.txt okunuyor...

   📊 Toplam Kamera Sayısı: 8
   ✅ Temiz (Ana) Dosya Sayısı: 8
   🗑️  Tespit Edilen 'Extra': 0 (0 olmalı)

   🔎 Örnek Dosya İsimleri (İlk 5):
      -> image08.png
      -> image07.png
      -> image06.png
      -> image05.png
      -> image04.png

----------------------------------------
✅ Tüm .bin dosyaları mevcut (images.bin, cameras.bin, points3D.bin)
   💾 Sparse Model Boyutu (points3D.bin): 0.28 MB

----------------------------------------
✅ Dense Point Cloud MEVCUT: /content/data/my_scene/colmap_output/dense_point_cloud.ply
   ⚖️  Dosya Boyutu: 8.31 MB
   👍 Boyut makul görünüyor.
🟢 ONAYLANDI: Veri seti %100 temiz ve eğitime hazır.


## 🎭 Cell 4: SAM2 Maske Oluşturma

YOLO + SAM2.1 ile otomatik maske oluşturur.

In [ ]:
%%bash
# --- 1) SAM2 temizle
pip uninstall -y sam2 SAM-2 || true
rm -rf /content/sam2

# --- 2) Clone + install (resmi öneri: pip install -e .)
git clone -q https://github.com/facebookresearch/sam2.git /content/sam2
cd /content/sam2
pip  install -e ".[notebooks]"   # jupyter/matplotlib bağımlılıkları dahil

# --- 3) Checkpoint indir (SAM2.1)
mkdir -p /content/sam2/checkpoints
wget  -O /content/sam2/checkpoints/sam2.1_hiera_large.pt \
  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

# --- 4) Hızlı smoke test
python - << 'PY'
from sam2.build_sam import build_sam2_video_predictor
print("SAM2 import OK")
PY

Obtaining file:///content/sam2
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for SAM-2 (pyproject.toml): started
  Building editable for SAM-2 (pyproject.toml): finished with status 'done'
  Created wheel for SAM-2: filename=sam_2-1.0-0.editable-cp311-cp311-linux_x86_64.whl size=13851 sha256=5ebda0864e8deaeee34f77ff000ef46323af7b090f2373c10af9ca6547ecc244
  Stored in directory: /tmp/pip-ephem-wheel-cache-bsqfq99b/wheels/76/dc/37/006d341f6080de50c00d031747ee8a1a03f3fb513175bce1c0
Successfully built SAM-2
SAM2 import OK


--2025-12-22 13:51:40--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.33.67.77, 13.33.67.73, 13.33.67.42, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.33.67.77|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 898083611 (856M) [application/vnd.snesdev-page-table]
Saving to: ‘/content/sam2/checkpoints/sam2.1_hiera_large.pt’

     0K .......... .......... .......... .......... ..........  0% 1.48M 9m40s
    50K .......... .......... .......... .......... ..........  0% 1.90M 8m36s
   100K .......... .......... .......... .......... ..........  0% 4.69M 6m45s
   150K .......... .......... .......... .......... ..........  0% 2.97M 6m16s
   200K .......... .......... .......... .......... ..........  0% 6.53M 5m27s
   250K .......... .......... .......... .......... ..........  0% 6.59M 4m54s
   300K .......... .......... .......... .......... ..

In [ ]:
import os, glob
from PIL import Image
import torch
from sam2.build_sam import build_sam2_video_predictor

ROOT = "/content/data/my_scene"              # <-- senin root
CAMS = [f"cam{i:02d}" for i in range(1,9)]     # cam00..cam07
SRC_BASE = os.path.join(ROOT, "added_environment")   # ör: frames_by_cam/cam00/*.png
DST_BASE = os.path.join(ROOT, "_sam2_frames")    # çıkış jpg frame klasörleri
OUT_BASE = os.path.join(ROOT, "masks_sam2")      # mask çıkışı

CKPT = "/content/sam2/checkpoints/sam2.1_hiera_large.pt"
CFG  = "configs/sam2.1/sam2.1_hiera_l.yaml"

predictor = build_sam2_video_predictor(CFG, CKPT, device="cuda")

def prep_frames(src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    frames = []
    for ext in ("*.png","*.jpg","*.jpeg","*.PNG","*.JPG","*.JPEG"):
        frames += glob.glob(os.path.join(src_dir, ext))
    frames = sorted(frames)
    assert frames, f"Frame yok: {src_dir}"
    for i,f in enumerate(frames):
        out = os.path.join(dst_dir, f"{i:05d}.jpg")
        Image.open(f).convert("RGB").save(out, quality=95)
    return len(frames)

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    for cam in CAMS:
        src_dir = os.path.join(SRC_BASE, cam)
        dst_dir = os.path.join(DST_BASE, cam)
        out_dir = os.path.join(OUT_BASE, cam)
        os.makedirs(out_dir, exist_ok=True)

        n = prep_frames(src_dir, dst_dir)

        state = predictor.init_state(video_path=dst_dir)
        predictor.reset_state(state)

        # TODO: burada prompt vermen şart:
        # - frame_idx=0’da YOLO bbox (x0,y0,x1,y1) ver
        # - predictor.add_new_points_or_box(...) çağır
        # - predictor.propagate_in_video(state) ile tüm framelere yay
        # - çıkan maskeleri out_dir/00000.png diye yaz

        print(cam, "OK, frames:", n)


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.15it/s]


cam01 OK, frames: 66


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.33it/s]


cam02 OK, frames: 66


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.33it/s]


...

cam06 OK, frames: 66


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.15it/s]


cam07 OK, frames: 66


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.27it/s]


cam08 OK, frames: 66


In [ ]:
from ultralytics import YOLO
import os
import numpy as np

# 1) YOLO modelin (senin notebooktaki path neyse onu koy)
yolo = YOLO("/content/yolov8m.pt")   # <-- düzelt

# 2) Hangi sınıfı segment edeceğiz?
# Modelinde sınıf adları varsa bunu kullan:
TARGET_CLASS_NAME = "person"  # <-- örnek: "person", "bag", "camera" vs.

DST_BASE = "/content/data/my_scene/_sam2_frames"  # senin jpg frame klasörün

def pick_bbox_from_yolo(img_path: str):
    r = yolo(img_path, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return None

    # class name -> class id bul
    names = r.names  # {id: "name"}
    target_ids = [cid for cid, n in names.items() if n == TARGET_CLASS_NAME]
    if not target_ids:
        raise ValueError(f"YOLO modelinde '{TARGET_CLASS_NAME}' diye class yok. names={names}")

    xyxy = r.boxes.xyxy.cpu().numpy()   # (N,4)
    conf = r.boxes.conf.cpu().numpy()   # (N,)
    cls  = r.boxes.cls.cpu().numpy().astype(int)  # (N,)

    # hedef sınıfa filtre
    mask = np.isin(cls, target_ids)
    if not mask.any():
        return None

    xyxy_f = xyxy[mask]
    conf_f = conf[mask]

    # en yüksek confidence bbox’u seç
    i = int(np.argmax(conf_f))
    return xyxy_f[i].tolist()  # [x0,y0,x1,y1]

BBOX_BY_CAM = {}
for cam in CAMS:
    img0 = os.path.join(DST_BASE, cam, "00000.jpg")
    bbox = pick_bbox_from_yolo(img0)
    if bbox is None:
        raise RuntimeError(f"{cam} frame0 YOLO detection yok: {img0}")
    BBOX_BY_CAM[cam] = bbox

print("BBOX_BY_CAM hazır:", BBOX_BY_CAM)


BBOX_BY_CAM hazır: {'cam01': [862.8431396484375, 59.83685302734375, 1144.0843505859375, 1030.540283203125], 'cam02': [790.1964111328125, 53.1405029296875, 1157.690673828125, 1028.44482421875], 'cam03': [733.1131591796875, 60.0640869140625, 1230.744384765625, 974.9313354492188], 'cam04': [724.8649291992188, 45.421142578125, 1177.550048828125, 1020.4345092773438], 'cam05': [781.567626953125, 60.098785400390625, 1066.4482421875, 1030.3677978515625], 'cam06': [727.1566772460938, 62.4686279296875, 1107.7802734375, 1019.4102172851562], 'cam07': [693.6357421875, 63.73516845703125, 1169.1171875, 967.0900268554688], 'cam08': [765.3072509765625, 68.21072387695312, 1216.759033203125, 1012.3613891601562]}


In [ ]:
import os
import numpy as np
import torch
from PIL import Image
import gc

# Yolları senin yapına göre ayarla
OUT_BASE = "/content/data/my_scene/masks_sam2"
DST_BASE = "/content/data/my_scene/_sam2_frames"
# Eğer BBOX_BY_CAM önceki adımda oluşmadıysa hata verecektir, o yüzden kontrol et
if 'BBOX_BY_CAM' not in globals():
    raise RuntimeError("⚠️ Önceki adımı (3. Adım - YOLO) çalıştırıp BBOX_BY_CAM sözlüğünü oluşturmalısın!")

print("="*60)
print("🚀 SAM2 Manuel Maskeleme (Düzeltilmiş 4. Adım)")
print("="*60)

def save_mask_png(mask_logits: np.ndarray, path: str):
    # Logits -> Binary Mask (0 veya 255)
    # Threshold genelde 0.0'dır (Sigmoid öncesi)
    mask_binary = (mask_logits > 0.0).astype(np.uint8) * 255
    # Sıkıştırma yaparak kaydet
    Image.fromarray(mask_binary[0, 0]).save(path)

def run_cam_safe(cam_name):
    print(f"🎥 İşleniyor: {cam_name}...")

    frames_dir = os.path.join(DST_BASE, cam_name)
    out_dir = os.path.join(OUT_BASE, cam_name)
    os.makedirs(out_dir, exist_ok=True)

    # YOLO'dan gelen kutu [x1, y1, x2, y2]
    box = np.array(BBOX_BY_CAM[cam_name], dtype=np.float32)

    # State başlatma
    inference_state = predictor.init_state(video_path=frames_dir)
    predictor.reset_state(inference_state)

    # 1. İlk Kareye Prompt (Kutu) Ver
    # SAM 2.1 API Güncellemesi: 'inference_state' ilk parametre
    _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=0,
        obj_id=1,
        box=box
    )

    # 2. Videoda Yayılım (Propagate)
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
        # Maskeyi kaydet
        # out_mask_logits shape: (1, H, W) veya (K, H, W)
        mask_path = os.path.join(out_dir, f"{out_frame_idx:05d}.png")

        # Binary'e çevir ve kaydet
        mask_arr = (out_mask_logits[0] > 0.0).cpu().numpy().astype(np.uint8) * 255

        # SAM2 maskeleri bazen (1, H, W) gelir, bazen (H, W).
        if mask_arr.ndim == 3:
            mask_arr = mask_arr[0]

        Image.fromarray(mask_arr).save(mask_path)

    # Hafıza Temizliği
    # predictor.reset_state(inference_state) # State'i temizle
    del inference_state
    torch.cuda.empty_cache()
    gc.collect()
    print(f"✅ {cam_name} tamamlandı.")

# Tüm kameraları işle
# Eğer CAMS listesi tanımlı değilse, klasörden bul
if 'CAMS' not in globals():
    CAMS = sorted([d for d in os.listdir(DST_BASE) if os.path.isdir(os.path.join(DST_BASE, d))])

for cam in CAMS:
    try:
        run_cam_safe(cam)
    except Exception as e:
        print(f"❌ HATA ({cam}): {e}")
        import traceback
        traceback.print_exc()

print("\n🎉 TÜM İŞLEMLER BİTTİ!")
print(f"📂 Maskeler şurada: {OUT_BASE}")

🚀 SAM2 Manuel Maskeleme (Düzeltilmiş 4. Adım)
🎥 İşleniyor: cam01...


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.31it/s]
/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video:  11%|█         | 7/66 [00:01<00:10,  5.41it/s]/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM

✅ cam01 tamamlandı.
🎥 İşleniyor: cam02...


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.33it/s]
/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video:   8%|▊         | 5/66 [00:00<00:10,  5.94it/s]/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM

✅ cam02 tamamlandı.
🎥 İşleniyor: cam03...


...

✅ cam06 tamamlandı.
🎥 İşleniyor: cam07...


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.02it/s]
/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video:   5%|▍         | 3/66 [00:00<00:08,  7.40it/s]/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM

✅ cam07 tamamlandı.
🎥 İşleniyor: cam08...


frame loading (JPEG): 100%|██████████| 66/66 [00:03<00:00, 20.06it/s]
/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(
propagate in video:  11%|█         | 7/66 [00:01<00:10,  5.43it/s]/content/sam2/sam2/sam2_video_predictor.py:786: UserWarning: Python version mismatch: module was compiled for Python 3.11, but the interpreter version is incompatible: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0].

Skipping the post-processing step due to the error above. You can still use SAM

✅ cam08 tamamlandı.

🎉 TÜM İŞLEMLER BİTTİ!
📂 Maskeler şurada: /content/data/my_scene/masks_sam2


## 👀 Cell 5: Maske Önizleme

Oluşturulan maskeleri kontrol edin.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import glob
import os
from ipywidgets import interact, IntSlider

print("="*60)
print("👀 Maske Önizleme (SAM2 Çıktıları)")
print("="*60)

# --- AYARLAR ---
# Orijinal görsellerin olduğu yer
SRC_BASE = "/content/data/my_scene/added_environment"
# Maskelerin olduğu yer (Senin son işlem sonucun)
MASK_BASE = "/content/data/my_scene/masks_sam2"

# Kamera klasörlerini bul
cam_folders = sorted(glob.glob(os.path.join(SRC_BASE, "cam*")))

if not cam_folders:
    print(f"❌ Kamera klasörü bulunamadı: {SRC_BASE}")
else:
    def preview_mask(camera_idx=0, frame_idx=0):
        cam_folder = cam_folders[camera_idx]
        cam_name = os.path.basename(cam_folder)

        # Orijinal Frame dosyalarını bul (PNG veya JPG)
        frame_files = sorted(glob.glob(os.path.join(cam_folder, "frame_*.jpg")))
        if not frame_files:
             frame_files = sorted(glob.glob(os.path.join(cam_folder, "*.jpg")))

        # Eğer JPG yoksa PNG dene
        if not frame_files:
            frame_files = sorted(glob.glob(os.path.join(cam_folder, "*.png")))

        if not frame_files:
            print(f"⚠️ {cam_name} içinde görsel bulunamadı.")
            return

        if frame_idx >= len(frame_files):
            print(f"Frame {frame_idx} sınır dışı (toplam {len(frame_files)})")
            return

        frame_path = frame_files[frame_idx]

        # --- MASKE YOLUNU BULMA (Kritik Düzeltme) ---
        # Senin maskelerin "masks_sam2/camXX/00000.png" formatında
        mask_folder = os.path.join(MASK_BASE, cam_name)
        # Frame indexine göre maske ismi (00000.png, 00001.png...)
        mask_name = f"{frame_idx:05d}.png"
        mask_path = os.path.join(mask_folder, mask_name)

        # Resmi yükle
        image = np.array(Image.open(frame_path).convert('RGB'))

        # Maskeyi yükle
        if os.path.exists(mask_path):
            mask = np.array(Image.open(mask_path).convert('L'))

            # Maske bazen görselden farklı boyutta olabilir (resize olduysa), eşitleyelim
            if mask.shape[:2] != image.shape[:2]:
                 mask = np.array(Image.open(mask_path).resize((image.shape[1], image.shape[0]), Image.NEAREST).convert('L'))

            # Overlay oluştur
            overlay = image.copy()
            # Foreground'u yeşile boya (Maske > 0 ise)
            # SAM2 çıktısı bazen binary (0-255) bazen logits olabilir, 128 güvenli eşiktir
            overlay[:,:,1] = np.where(mask > 128, np.minimum(overlay[:,:,1] + 100, 255), overlay[:,:,1])
            mask_status = "✅ Maske Mevcut"
        else:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
            overlay = image
            mask_status = f"❌ Maske Yok: {mask_name}"

        # Görselleştir
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(image)
        axes[0].set_title(f"{cam_name} - Frame {frame_idx}\nOrijinal")
        axes[0].axis('off')

        axes[1].imshow(mask, cmap='gray')
        axes[1].set_title(f"Maske\n({mask_status})")
        axes[1].axis('off')

        axes[2].imshow(overlay)
        axes[2].set_title("Overlay\n(Yeşil = İnsan)")
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

    # Toplam frame sayısını ilk kameradan alalım
    first_cam_path = cam_folders[0]
    num_frames = len(glob.glob(os.path.join(first_cam_path, "*.jpg")) + glob.glob(os.path.join(first_cam_path, "*.png")))

    # Interaktif önizleme
    print(f"\n📸 {len(cam_folders)} kamera tespit edildi.")
    print(f"🖼️ Yaklaşık {num_frames} kare (frame) var.\n")

    interact(
        preview_mask,
        camera_idx=IntSlider(min=0, max=len(cam_folders)-1, step=1, value=0, description='Kamera No'),
        frame_idx=IntSlider(min=0, max=num_frames-1, step=1, value=0, description='Frame No')
    )

print("\n📝 Eğer yeşil alan insanı doğru kaplıyorsa Cell 6'ya (Eğitim) geçebilirsin.")

👀 Maske Önizleme (SAM2 Çıktıları)

📸 8 kamera tespit edildi.
🖼️ Yaklaşık 66 kare (frame) var.



interactive(children=(IntSlider(value=0, description='Kamera No', max=7), IntSlider(value=0, description='Fram…


📝 Eğer yeşil alan insanı doğru kaplıyorsa Cell 6'ya (Eğitim) geçebilirsin.


## ⚙️ Cell 6: Eğitim Konfigürasyonu

Eğitim parametrelerini ayarlayın.

In [ ]:
import os
import shutil

print("="*60)
print("⚙️  Eğitim Konfigürasyonu ve Veri Hazırlığı")
print("="*60)

# ==============================================================================
# 1. VERİ SETİ HAZIRLIĞI (Maskeleri ve Görselleri Birleştirme)
# ==============================================================================
# COLMAP çıktısının olduğu yer (Ana dataset kökü burası olacak)
DATASET_PATH = "/content/data/my_scene/colmap_output"
# Maskelerin olduğu yer
MASK_SOURCE = "/content/data/my_scene/masks_sam2"
# Orijinal görsellerin olduğu yer
IMAGE_SOURCE = "/content/data/my_scene/added_environment"

print(f"📂 Veri Seti Yolu: {DATASET_PATH}")

# A) MASKELERİ BAĞLA: Eğitim kodunun maskeleri otomatik tanıması için
# 'masks' klasörü dataset içinde olmalı. Kopyalamak yerine Symlink yapıyoruz (Yer kaplamaz).
target_mask_path = os.path.join(DATASET_PATH, "masks")
if os.path.exists(target_mask_path):
    if os.path.islink(target_mask_path):
        os.unlink(target_mask_path) # Eski linki kaldır
    elif os.path.isdir(target_mask_path):
        shutil.rmtree(target_mask_path) # Eski klasörü sil

try:
    os.symlink(MASK_SOURCE, target_mask_path)
    print(f"   ✅ Maskeler bağlandı: {MASK_SOURCE} -> {target_mask_path}")
except OSError as e:
    # Symlink hatası olursa kopyalamayı dene
    print(f"   ⚠️ Symlink yapılamadı, kopyalanıyor... ({e})")
    shutil.copytree(MASK_SOURCE, target_mask_path)

# B) GÖRSELLERİ KONTROL ET
# Colmap output içinde 'images' klasörü olmayabilir, onu da bağlayalım.
target_image_path = os.path.join(DATASET_PATH, "images")
if not os.path.exists(target_image_path):
    try:
        os.symlink(IMAGE_SOURCE, target_image_path)
        print(f"   ✅ Görseller bağlandı: {IMAGE_SOURCE} -> {target_image_path}")
    except OSError:
        shutil.copytree(IMAGE_SOURCE, target_image_path)

# ==============================================================================
# 2. EĞİTİM PRESETLERİ (A100 İçin Güçlendirildi)
# ==============================================================================
PRESETS = {
    "standard": {
        "iterations": 30000,
        "densify_until_iter": 15000,
        "coarse_iterations": 3000,
        "w_fg": 1.0, "w_bg": 0.1,
        "net_width": 64,
        "desc": "Standart (30k iterasyon)"
    },
    "high_quality": {
        "iterations": 50000,
        "densify_until_iter": 25000,
        "coarse_iterations": 5000,
        "w_fg": 2.0, "w_bg": 0.05, # İnsana daha çok odaklan
        "net_width": 128, # Daha geniş ağ (daha iyi deformasyon)
        "desc": "Yüksek Kalite (50k iter, İnsan odaklı)"
    },
    "ultra_quality": {
        "iterations": 80000, # A100 Gücü!
        "densify_until_iter": 40000,
        "coarse_iterations": 8000,
        "w_fg": 2.0, "w_bg": 0.1, # Arka planı neredeyse yok say, insana aban
        "net_width": 128,
        "desc": "ULTRA Kalite (80k iter, A100 Özel)"
    }
}

# --- AYARLAR ---
# Tavsiyem: A100 olduğu için 'high_quality' veya 'ultra_quality' seçmen.
SELECTED_PRESET = "ultra_quality" # @param ["standard", "high_quality", "ultra_quality"]

cfg = PRESETS[SELECTED_PRESET]

# Parametre Değişkenleri (Cell 7 bunları kullanacak)
ITERATIONS = cfg["iterations"]
DENSIFY_UNTIL = cfg["densify_until_iter"]
COARSE_ITERS = cfg["coarse_iterations"]
NET_WIDTH = cfg["net_width"]

# Maske Ağırlıkları (Mask Weighted Loss)
USE_MASK = True
W_FG = cfg["w_fg"]  # Ön plan ağırlığı
W_BG = cfg["w_bg"]  # Arka plan ağırlığı

print("\n" + "="*60)
print(f"🎯 Seçilen Mod: {SELECTED_PRESET.upper()}")
print(f"   📝 {cfg['desc']}")
print("="*60)
print(f"   🔄 Toplam İterasyon: {ITERATIONS}")
print(f"   🧠 Ağ Genişliği: {NET_WIDTH}")
print(f"   🎭 Maske Kullanımı: {USE_MASK}")
print(f"      - Ön Plan Ağırlığı: {W_FG} (İnsan)")
print(f"      - Arka Plan Ağırlığı: {W_BG} (Çevre)")
print(f"   📂 Dataset Yolu: {DATASET_PATH}")
print("\n📝 Sonraki adım: Cell 7 ile eğitimi başlatın (Loglar açık olacak)")

⚙️  Eğitim Konfigürasyonu ve Veri Hazırlığı
📂 Veri Seti Yolu: /content/data/my_scene/colmap_output
   ✅ Maskeler bağlandı: /content/data/my_scene/masks_sam2 -> /content/data/my_scene/colmap_output/masks

🎯 Seçilen Mod: ULTRA_QUALITY
   📝 ULTRA Kalite (80k iter, A100 Özel)
   🔄 Toplam İterasyon: 80000
   🧠 Ağ Genişliği: 128
   🎭 Maske Kullanımı: True
      - Ön Plan Ağırlığı: 2.0 (İnsan)
      - Arka Plan Ağırlığı: 0.1 (Çevre)
   📂 Dataset Yolu: /content/data/my_scene/colmap_output

📝 Sonraki adım: Cell 7 ile eğitimi başlatın (Loglar açık olacak)


## 🚀 Cell 7: Eğitim

Model eğitimini başlatır. Eğitim lokal diskte yapılır, sonunda Drive'a kopyalanır.

In [ ]:
import os
import struct
import shutil
import glob

print("="*60)
print("🎯 HEDEF ODAKLI PATH DÜZELTİCİ (DOĞRU ADRES)")
print("="*60)

# 1. EĞİTİMDE KULLANILAN GERÇEK PATH
# Cell 7'de kullanılan path burası:
REAL_DATASET_PATH = "/content/data/my_scene/colmap_output"
IMAGES_DIR = os.path.join(REAL_DATASET_PATH, "images")
SPARSE_DIR = os.path.join(REAL_DATASET_PATH, "sparse/0")
IMAGES_BIN = os.path.join(SPARSE_DIR, "images.bin")

print(f"📂 Veri Seti: {REAL_DATASET_PATH}")
print(f"📄 Hedef Dosya: {IMAGES_BIN}")

if not os.path.exists(IMAGES_BIN):
    print("❌ HATA: images.bin bulunamadı! Yol yanlış olabilir.")
    # Belki sparse klasörü direkt köktedir, kontrol et
    alt_path = os.path.join(REAL_DATASET_PATH, "sparse/images.bin")
    if os.path.exists(alt_path):
        IMAGES_BIN = alt_path
        print(f"   ⚠️ Dosya şurada bulundu ve güncellendi: {IMAGES_BIN}")
    else:
        raise FileNotFoundError("images.bin hiçbir yerde bulunamadı.")

# 2. GERÇEK KLASÖR YAPISINI TESPİT ET
# Diskte 'cam01', 'cam02' mi var yoksa 'image01' mi?
folder_candidates = sorted([f for f in os.listdir(IMAGES_DIR) if os.path.isdir(os.path.join(IMAGES_DIR, f))])
print(f"📂 Diskteki Klasörler: {folder_candidates}")

# İlk dosya ismini öğren (frame_00001.png mi 00000.png mi?)
sample_folder = os.path.join(IMAGES_DIR, folder_candidates[0])
sample_files = sorted([f for f in os.listdir(sample_folder) if f.endswith(('.png', '.jpg'))])
if not sample_files:
    raise ValueError("❌ Klasörler boş!")
target_filename = sample_files[0] # örn: frame_00001.png
print(f"🖼️ Örnek Dosya Adı: {target_filename}")

# 3. YARDIMCI FONKSİYONLAR
def read_next_bytes(fid, num_bytes, format_char_sequence, endian_character="<"):
    data = fid.read(num_bytes)
    return struct.unpack(endian_character + format_char_sequence, data)

def write_next_bytes(fid, *args):
    fmt = "<" + args[-1]
    data = args[:-1]
    if len(data) == 1 and isinstance(data[0], (list, tuple)):
        data = data[0]
    fid.write(struct.pack(fmt, *data))

# 4. DOSYAYI OKU VE HAFIZAYA AL
print("\n🔄 Dosya okunuyor...")
images_data = []
with open(IMAGES_BIN, "rb") as fid:
    num_reg_images = read_next_bytes(fid, 8, "Q")[0]
    print(f"   Kayıtlı Kamera Sayısı: {num_reg_images}")

    for _ in range(num_reg_images):
        props = read_next_bytes(fid, 64, "I4d3dI")
        image_id = props[0]
        name = ""
        while True:
            char = fid.read(1)
            if char == b"\x00": break
            name += char.decode("utf-8")

        num_p2d = read_next_bytes(fid, 8, "Q")[0]
        points_data = fid.read(num_p2d * 24) # Veriyi atla ama sakla

        images_data.append({
            "props": props,
            "name": name,
            "num_p2d": num_p2d,
            "points": points_data,
            "id": image_id
        })

# ID'ye göre sırala (Eşleştirme için kritik)
images_data.sort(key=lambda x: x["id"])

# 5. VERİLERİ DÜZENLE (YAMA)
print("\n🛠️ Düzeltme Uygulanıyor...")
# Eğer diskteki klasör sayısı ile kayıttaki sayı tutuyorsa sırayla eşleştir
if len(images_data) == len(folder_candidates):
    for i, img in enumerate(images_data):
        real_folder = folder_candidates[i] # örn: cam01

        # ESKİ: frame_00001.png veya image01.png
        # YENİ: cam01/frame_00001.png

        new_name = f"{real_folder}/{target_filename}"
        print(f"   ID {img['id']}: {img['name']}  --->  {new_name}")
        img["name"] = new_name
else:
    print("⚠️ SAYI UYUŞMAZLIĞI! Otomatik eşleştirme riskli olabilir.")
    print(f"   Kayıt: {len(images_data)} vs Disk: {len(folder_candidates)}")
    print("   Mevcut isimleri 'camXX/...' formatına çevirmeyi deniyorum...")

    # Yedek plan: Mevcut isme bakıp klasör uydurmak
    for i, img in enumerate(images_data):
        # image01.png -> cam01/frame...
        # frame_00001.png -> cam01/frame... (Sıraya göre)
        if i < len(folder_candidates):
            new_name = f"{folder_candidates[i]}/{target_filename}"
            print(f"   ID {img['id']}: {img['name']}  --->  {new_name}")
            img["name"] = new_name

# 6. KAYDET
# Önce yedek al
shutil.copy(IMAGES_BIN, IMAGES_BIN + ".bak")
print(f"\n💾 Orijinal yedeklendi: {IMAGES_BIN}.bak")

with open(IMAGES_BIN, "wb") as fid:
    write_next_bytes(fid, len(images_data), "Q")
    for img in images_data:
        write_next_bytes(fid, *img["props"], "I4d3dI")
        fid.write(img["name"].encode("utf-8") + b"\x00")
        write_next_bytes(fid, img["num_p2d"], "Q")
        fid.write(img["points"])

print(f"✅ Dosya başarıyla güncellendi: {IMAGES_BIN}")
print("\n🚀 ŞİMDİ EĞİTİMİ BAŞLATABİLİRSİN (Cell 7)")

🎯 HEDEF ODAKLI PATH DÜZELTİCİ (DOĞRU ADRES)
📂 Veri Seti: /content/data/my_scene/colmap_output
📄 Hedef Dosya: /content/data/my_scene/colmap_output/sparse/0/images.bin
📂 Diskteki Klasörler: ['cam01', 'cam02', 'cam03', 'cam04', 'cam05', 'cam06', 'cam07', 'cam08']
🖼️ Örnek Dosya Adı: frame_00001.png

🔄 Dosya okunuyor...
   Kayıtlı Kamera Sayısı: 8

🛠️ Düzeltme Uygulanıyor...
   ID 33: cam01/frame_00001.png  --->  cam01/frame_00001.png
   ID 34: cam02/frame_00001.png  --->  cam02/frame_00001.png
   ID 35: cam03/frame_00001.png  --->  cam03/frame_00001.png
   ID 36: cam04/frame_00001.png  --->  cam04/frame_00001.png
   ID 37: cam05/frame_00001.png  --->  cam05/frame_00001.png
   ID 38: cam06/frame_00001.png  --->  cam06/frame_00001.png
   ID 39: cam07/frame_00001.png  --->  cam07/frame_00001.png
   ID 40: cam08/frame_00001.png  --->  cam08/frame_00001.png

💾 Orijinal yedeklendi: /content/data/my_scene/colmap_output/sparse/0/images.bin.bak
✅ Dosya başarıyla güncellendi: /content/data/my_scene

In [ ]:
import os
import re

print("="*60)
print("🧬 KAYNAK KOD AMELİYATI (DATASET READER PATCH)")
print("="*60)

# 1. HEDEF DOSYA
repo_path = "/content/4DGaussians-Enhanced"
target_file = os.path.join(repo_path, "scene", "dataset_readers.py")

if not os.path.exists(target_file):
    print("❌ HATA: Hedef dosya bulunamadı!")
    # Alternatif kontrol
    alt_repo = "/content/4DGaussians"
    if os.path.exists(os.path.join(alt_repo, "scene", "dataset_readers.py")):
        target_file = os.path.join(alt_repo, "scene", "dataset_readers.py")
        print(f"   ⚠️ Alternatif repo bulundu: {target_file}")
    else:
        raise FileNotFoundError("dataset_readers.py hiçbir yerde yok.")

print(f"📄 Hedef: {target_file}")

# 2. DOSYAYI OKU
with open(target_file, "r") as f:
    original_code = f.read()

# 3. SORUNLU SATIRI TESPİT ET VE DÜZELT
# Hedef: os.path.join(images_folder, os.path.basename(...)) yapısını bulup
#        os.path.join(images_folder, ...) haline getirmek.

# Regex ile esnek arama
# Bu regex, parantez içindeki değişken adını (capture group 1) yakalar.
pattern = r'os\.path\.join\s*\(\s*images_folder\s*,\s*os\.path\.basename\s*\(([^)]+)\)\s*\)'

match = re.search(pattern, original_code)

if match:
    print("\n🐛 KISITLAYICI KOD BULUNDU!")
    old_code = match.group(0)
    print(f"   Eski: {old_code}")

    # Değişken adı (örn: image.name veya extr.name)
    var_name = match.group(1)

    # Yeni kod (basename fonksiyonunu çıkarıyoruz)
    new_code = f"os.path.join(images_folder, {var_name})"
    print(f"   Yeni: {new_code}")

    # Değiştir
    patched_code = original_code.replace(old_code, new_code)

    # Kaydet
    with open(target_file, "w") as f:
        f.write(patched_code)

    print("\n✅ YAMA BAŞARIYLA UYGULANDI.")
    print("   Kod artık alt klasörleri (camXX/...) olduğu gibi okuyacak.")

else:
    # Eğer regex bulamazsa, manuel string replace deneyelim (B planı)
    print("⚠️ Regex eşleşmedi, B planı (Manuel Replace) deneniyor...")

    replacements = [
        ("os.path.join(images_folder, os.path.basename(image.name))", "os.path.join(images_folder, image.name)"),
        ("os.path.join(images_folder, os.path.basename(extr.name))", "os.path.join(images_folder, extr.name)")
    ]

    patched = False
    for old, new in replacements:
        if old in original_code:
            original_code = original_code.replace(old, new)
            print(f"   ✏️ Düzeltildi: {old} -> {new}")
            patched = True

    if patched:
        with open(target_file, "w") as f:
            f.write(original_code)
        print("\n✅ YAMA BAŞARIYLA UYGULANDI.")
    else:
        print("\n❌ HATA: Değiştirilecek kod bloğu bulunamadı. Dosya zaten yamalı olabilir mi?")

print("\n🚀 SONUÇ: Şimdi Cell 7'yi çalıştırıp eğitimi başlat!")

🧬 KAYNAK KOD AMELİYATI (DATASET READER PATCH)
📄 Hedef: /content/4DGaussians-Enhanced/scene/dataset_readers.py

🐛 KISITLAYICI KOD BULUNDU!
   Eski: os.path.join(images_folder, os.path.basename(extr.name))
   Yeni: os.path.join(images_folder, extr.name)

✅ YAMA BAŞARIYLA UYGULANDI.
   Kod artık alt klasörleri (camXX/...) olduğu gibi okuyacak.

🚀 SONUÇ: Şimdi Cell 7'yi çalıştırıp eğitimi başlat!


In [ ]:
import os
import sys
import time
import shutil
import subprocess
import glob
from datetime import datetime

print("="*60)
print("🛡️ CELL 7: FINAL EĞİTİM BAŞLATICI (DÜZELTİLMİŞ & GÜÇLENDİRİLMİŞ)")
print("="*60)

# ==============================================================================
# 1. KÜTÜPHANELERİ GARANTİLİ YÜKLE
# ==============================================================================
print(f"🐍 Python Yolu: {sys.executable}")
print("📦 Bağımlılıklar kontrol ediliyor...\n")

critical_packages = ["open3d", "plyfile", "lpips", "tqdm", "opencv-python"]
for pkg in critical_packages:
    try:
        subprocess.check_call([sys.executable, "-c", f"import {pkg.split('-')[0].replace('opencv', 'cv2')}"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except:
        print(f"   ⬇️  Yükleniyor: {pkg}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg], stdout=subprocess.DEVNULL)
            print(f"      ✅ Yüklendi: {pkg}")
        except:
            print(f"      ❌ {pkg} yüklenemedi!")
print("   ✅ Tüm kütüphaneler hazır.")

# ==============================================================================
# 2. AYARLAR VE YOLLAR
# ==============================================================================
if 'DATASET_PATH' not in globals():
    TRAIN_SOURCE = "/content/data/my_scene/colmap_output"
else:
    TRAIN_SOURCE = DATASET_PATH

# Çıktı Klasörü
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
LOCAL_OUTPUT = f"/content/output/4DGS_{timestamp}"
SAVE_ITERS = [7000, 15000, 30000, 50000, ITERATIONS]

# ==============================================================================
# 3. 🧪 MASKE TURNUSOL TESTİ (GELİŞMİŞ EŞLEŞTİRME)
# ==============================================================================
print("\n" + "="*60)
print("🧪 MASKE ENTEGRASYON TESTİ")
print("="*60)

if USE_MASK:
    mask_root = os.path.join(TRAIN_SOURCE, "masks")
    img_root = os.path.join(TRAIN_SOURCE, "images")

    # Görselleri bul (JPG veya PNG)
    sample_images = sorted(glob.glob(os.path.join(img_root, "**", "*.jpg"), recursive=True))[:5]
    if not sample_images:
        sample_images = sorted(glob.glob(os.path.join(img_root, "**", "*.png"), recursive=True))[:5]

    if sample_images:
        print(f"   🔬 Örneklem Kontrolü ({len(sample_images)} dosya):")
        match_count = 0

        for img_path in sample_images:
            rel_path = os.path.relpath(img_path, img_root) # örn: cam01/frame_00001.png
            folder = os.path.dirname(rel_path)              # örn: cam01
            fname = os.path.basename(rel_path)              # örn: frame_00001.png
            fname_no_ext = os.path.splitext(fname)[0]       # örn: frame_00001

            # Olası Maske İsimleri (Senaryolar)
            candidates = [
                fname.replace(".jpg", ".png"),                     # frame_00001.png (Birebir)
                fname_no_ext + ".png",                             # frame_00001.png (PNG ise)
                fname_no_ext.split("_")[-1] + ".png" if "_" in fname_no_ext else "JUNK", # 00001.png (Sayısal)
                f"{int(fname_no_ext.split('_')[-1]):05d}.png" if "_" in fname_no_ext and fname_no_ext.split("_")[-1].isdigit() else "JUNK" # 00001.png (Formatlı)
            ]

            found = False
            for cand in candidates:
                if cand == "JUNK": continue
                mask_full_path = os.path.join(mask_root, folder, cand)
                if os.path.exists(mask_full_path):
                    print(f"      ✅ Eşleşme: {fname} -> {cand}")
                    found = True
                    match_count += 1
                    break

            if not found:
                print(f"      ⚠️  Maske yok: {rel_path} (Denenenler: {candidates})")

        if match_count > 0:
            print("\n   ✅ TEST BAŞARILI: Maskeler algılandı.")
            print(f"   ⚖️  Ağırlıklar: FG={W_FG} | BG={W_BG}")
        else:
            print("\n❌ HATA: Hiçbir maske eşleşmedi!")
            print("   Lütfen Cell 6'da maskelerin 'masks_sam2' klasörüne doğru kopyalandığından emin olun.")
            sys.exit(1)
    else:
        print("❌ Dataset içinde görüntü dosyası bulunamadı!")
        sys.exit(1)
else:
    print("⚠️ USE_MASK = False. Maskesiz eğitim.")

# ==============================================================================
# 4. KOMUT OLUŞTURMA & ÇALIŞTIRMA
# ==============================================================================
work_dir = "/content/4DGaussians-Enhanced"

# Argüman listesi
cmd_args = [
    sys.executable, "train.py",
    "--source_path", TRAIN_SOURCE,
    "--model_path", LOCAL_OUTPUT,
    "--images", "images",
    "--iterations", str(ITERATIONS),
    "--coarse_iterations", str(COARSE_ITERS),
    "--densify_until_iter", str(DENSIFY_UNTIL),
    "--net_width", str(NET_WIDTH),
    "--batch_size", "1",
    "--resolution", "1",
    "--save_iterations", *map(str, SAVE_ITERS)
    # NOT: '--quiet False' parametresi SİLİNDİ (Hataya sebep oluyordu)
    # Parametre vermemek zaten logları açar.
]

if USE_MASK:
    cmd_args.append("--use_mask_loss")
    cmd_args.append("--w_fg"); cmd_args.append(str(W_FG))
    cmd_args.append("--w_bg"); cmd_args.append(str(W_BG))

print("\n" + "="*60)
print(f"🚀 EĞİTİM START ALIYOR (A100 - {ITERATIONS} Iterasyon)")
print("="*60)
print(f"📂 Çıktı: {LOCAL_OUTPUT}")

start_time = time.time()

# Subprocess ile çalıştır
process = subprocess.Popen(
    cmd_args,
    cwd=work_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

# LOG OKUMA (Eğitim başladığında burası akacak)
for line in process.stdout:
    print(line, end="")

process.wait()

# ==============================================================================
# 5. BİTİŞ & YEDEKLEME
# ==============================================================================
duration = time.time() - start_time
hours = duration // 3600
minutes = (duration % 3600) // 60

print("\n" + "="*60)
if process.returncode == 0:
    print(f"✅ EĞİTİM TAMAMLANDI! (Süre: {int(hours)}sa {int(minutes)}dk)")

    DRIVE_ROOT = "/content/drive/MyDrive/4DGS_project/output"
    drive_target = os.path.join(DRIVE_ROOT, f"scene_{timestamp}")
    print(f"📤 Drive Yedekleme: {drive_target}")

    try:
        if not os.path.exists(DRIVE_ROOT): os.makedirs(DRIVE_ROOT)
        shutil.copytree(LOCAL_OUTPUT, drive_target)
        print("✅ Başarıyla yedeklendi.")
    except Exception as e:
        print(f"❌ Yedekleme hatası: {e}")
else:
    print(f"❌ EĞİTİM HATA İLE SONLANDI (Kod: {process.returncode})")

print("="*60)

Streaming output truncated to the last 5000 lines.
Training progress: 100%|██████████| 80000/80000 [2:47:00<00:00,  7.98it/s, Loss=0.0018994, psnr=48.57, point=265166]
data loading done [22/12 15:37:30]

[ITER 3000] Evaluating test: L1 0.0837966650724411 PSNR 15.692878723144531 [22/12 15:42:47]

[ITER 3000] Evaluating train: L1 0.005239250275361187 PSNR 34.76992595897001 [22/12 15:42:54]
reset opacity [22/12 15:42:54]
reset opacity [22/12 15:48:24]

[ITER 7000] Evaluating test: L1 0.09106110781431198 PSNR 15.426862716674805 [22/12 15:50:24]

[ITER 7000] Evaluating train: L1 0.002909019575728213 PSNR 41.499301461612475 [22/12 15:50:27]

[ITER 7000] Saving Gaussians [22/12 15:50:27]
reset opacity [22/12 15:54:20]
reset opacity [22/12 16:00:30]

[ITER 14000] Evaluating test: L1 0.09185090661048889 PSNR 15.379354476928711 [22/12 16:04:52]

[ITER 14000] Evaluating train: L1 0.0024364181684658807 PSNR 44.50333830889534 [22/12 16:04:55]

[ITER 15000] Saving Gaussians [22/12 16:07:02]
reset op

## 🎥 Cell 8: Render

Eğitilmiş modelden video render eder.

In [ ]:
import glob
from IPython.display import Video, display

print("="*60)
print("🎥 Video Render")
print("="*60)

# Render komutu
render_cmd = f"""python /content/4DGaussians-Enhanced/render.py \
    --source_path {LOCAL_DATA} \
    --model_path {LOCAL_OUTPUT} \
    --iteration {ITERATIONS}"""

print(f"\n📝 Komut:")
print(render_cmd)
print()

!{render_cmd}

# Render edilen videoyu bul
video_files = glob.glob(os.path.join(LOCAL_OUTPUT, "**/*.mp4"), recursive=True)

if video_files:
    print("\n" + "="*60)
    print("✅ Render tamamlandı!")
    print("="*60)
    print(f"\n📹 Video: {video_files[0]}")

    # Videoyu göster
    print("\n📺 Video oynatılıyor...\n")
    display(Video(video_files[0], width=800))

    # Drive'a kopyala
    scene_name = os.path.basename(LOCAL_DATA)
    drive_output = os.path.join(OUTPUT_BASE, scene_name)

    print(f"\n📤 Video Drive'a kopyalanıyor: {drive_output}")
    # Model zaten kopyalandı, sadece render klasörünü güncelle
    render_dir_local = os.path.dirname(video_files[0])
    render_dir_drive = os.path.join(drive_output, os.path.basename(render_dir_local))

    if os.path.exists(render_dir_drive):
        shutil.rmtree(render_dir_drive)
    shutil.copytree(render_dir_local, render_dir_drive)
    print(f"✅ Video Drive'a kopyalandı")
else:
    print("\n⚠️  Video dosyası bulunamadı. Çıktı klasörünü kontrol edin.")

print("\n📝 Sonraki adım: Cell 9 ile PLY export yapın (opsiyonel)")

## 💾 Cell 9: PLY Export (Opsiyonel)

Frame başına 3D Gaussian point cloud'ları export eder.

In [ ]:
import os
import sys
import shutil
import glob
import subprocess

print("="*60)
print("💾 CELL 9: PLY EXPORT (NET_WIDTH DÜZELTİLMİŞ)")
print("="*60)

EXPORT_PLY = True  # @param {type:"boolean"}

# --- YOLLARIN TANIMLANMASI ---
# 1. Dataset
if 'DATASET_PATH' not in globals():
    source_path = "/content/data/my_scene/colmap_output"
else:
    source_path = DATASET_PATH

# 2. Model Çıktısı
if 'LOCAL_OUTPUT' not in globals() or not os.path.exists(LOCAL_OUTPUT):
    output_root = "/content/output"
    if os.path.exists(output_root):
        all_outputs = sorted(glob.glob(os.path.join(output_root, "4DGS_*")))
        if all_outputs:
            LOCAL_OUTPUT = all_outputs[-1]
            print(f"⚠️ LOCAL_OUTPUT güncellendi: {LOCAL_OUTPUT}")
        else:
            print("❌ HATA: Çıktı klasörü bulunamadı!")
            EXPORT_PLY = False
    else:
        print("❌ HATA: /content/output bulunamadı!")
        EXPORT_PLY = False

model_path = LOCAL_OUTPUT

# 3. Model Parametreleri (Hata Çözücü)
# Hataya göre checkpoint 128 boyutunda. Bunu script'e bildirmeliyiz.
NET_WIDTH = 128

if EXPORT_PLY:
    print(f"📂 Kaynak: {source_path}")
    print(f"📂 Model: {model_path}")
    print(f"🧠 Ağ Genişliği: {NET_WIDTH} (Checkpoint ile eşleşmeli)")

    # Çalışma dizini
    work_dir = "/content/4DGaussians-Enhanced"

    # --- DÜZELTME BURADA ---
    # '--net_width 128' eklendi.
    cmd_args = [
        sys.executable, "export_perframe_3DGS.py",
        "--source_path", source_path,
        "--model_path", model_path,
        "--iteration", str(ITERATIONS),
        "--skip_video",
        "--net_width", str(NET_WIDTH) # <--- KRİTİK EKLEME
    ]

    print("\n📝 Export işlemi başlıyor (Bu işlem biraz sürebilir)...")
    print("-" * 60)

    try:
        process = subprocess.Popen(
            cmd_args,
            cwd=work_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            universal_newlines=True
        )

        for line in process.stdout:
            print(line, end="")

        process.wait()

        if process.returncode != 0:
            print(f"\n❌ Export hata kodu: {process.returncode}")
        else:
            print("\n✅ Export komutu başarıyla tamamlandı.")

    except Exception as e:
        print(f"\n❌ Hata: {e}")

    # --- KONTROL VE DRIVE YEDEKLEME ---
    ply_source_dir = os.path.join(model_path, "per_frame_ply")

    if os.path.exists(ply_source_dir):
        ply_files = glob.glob(os.path.join(ply_source_dir, "*.ply"))

        print("\n" + "="*60)
        print(f"💾 {len(ply_files)} karelik 4D PLY serisi bulundu!")
        print(f"📁 Konum: {ply_source_dir}")

        # Drive Hedefi
        model_name = os.path.basename(model_path)
        drive_ply_dir = os.path.join("/content/drive/MyDrive/4DGS_project/output", model_name, "ply_sequence")

        print(f"\n📤 Drive'a aktarılıyor: {drive_ply_dir}")

        if not os.path.exists(os.path.dirname(drive_ply_dir)):
            os.makedirs(os.path.dirname(drive_ply_dir), exist_ok=True)

        if os.path.exists(drive_ply_dir):
            shutil.rmtree(drive_ply_dir)

        shutil.copytree(ply_source_dir, drive_ply_dir)
        print("✅ Kopyalama tamamlandı!")
        print("="*60)
        print("💡 İPUCU: Dosyalar Drive'a yüklendiğinde 'SuperSplat' sitesine sürükleyip bırakarak izleyebilirsin.")

    else:
        print(f"\n⚠️ 'per_frame_ply' klasörü oluşmadı.")
        print("   Loglarda 'traceback' hatası var mı kontrol edin.")

else:
    print("⏭️  Export atlandı.")

💾 CELL 9: PLY EXPORT (NET_WIDTH DÜZELTİLMİŞ)
📂 Kaynak: /content/data/my_scene/colmap_output
📂 Model: /content/output/4DGS_20251222_1530
🧠 Ağ Genişliği: 128 (Checkpoint ile eşleşmeli)

📝 Export işlemi başlıyor (Bu işlem biraz sürebilir)...
------------------------------------------------------------
  File "/content/4DGaussians-Enhanced/export_perframe_3DGS.py", line 113
    points, scales_final, rotations_final, opacity_final, shs_final = get_state_at_time(gaussians, viewpoint)
    ^^^^^^
IndentationError: expected an indented block after 'for' statement on line 110

❌ Export hata kodu: 1

⚠️ 'per_frame_ply' klasörü oluşmadı.
   Loglarda 'traceback' hatası var mı kontrol edin.


In [ ]:
import os
import sys
import shutil
import glob
import subprocess

print("="*60)
print("🚑 SCRIPT TAMİRİ VE EXPORT BAŞLATMA")
print("="*60)

# --- 1. SCRIPT AMELİYATI (PATCH) ---
script_path = "/content/4DGaussians-Enhanced/export_perframe_3DGS.py"
print(f"🔧 Hedef Dosya: {script_path}")

with open(script_path, "r") as f:
    code = f.read()

# Eski Hatalı Satır
old_line = "for index, viewpoint in enumerate(scene.getTestCameras()):"

# Yeni Mantıklı Blok
# Önce Test'e bak, boşsa Train'i al.
new_block = """
    # YAMA: Kameraları akıllı seç
    cameras = scene.getTestCameras()
    if len(cameras) == 0:
        print("⚠️ Test kamerası bulunamadı, Eğitim kameraları (Train) kullanılıyor...")
        cameras = scene.getTrainCameras()

    # Sıralı olması için isme veya zamana göre sıralayalım (Opsiyonel ama iyi olur)
    # cameras.sort(key=lambda x: x.uid)

    for index, viewpoint in enumerate(cameras):
"""

if old_line in code:
    print("   🐛 Hatalı döngü bulundu, düzeltiliyor...")
    new_code = code.replace(old_line, new_block)

    with open(script_path, "w") as f:
        f.write(new_code)
    print("   ✅ Yama başarıyla uygulandı!")
else:
    print("   ℹ️ Yama zaten uygulanmış veya kod farklı.")

# --- 2. EXPORT İŞLEMİNİ BAŞLAT ---
print("\n" + "="*60)
print("🎬 EXPORT START")
print("="*60)

# Yolları Tanımla
if 'DATASET_PATH' not in globals():
    source_path = "/content/data/my_scene/colmap_output"
else:
    source_path = DATASET_PATH

if 'LOCAL_OUTPUT' not in globals() or not os.path.exists(LOCAL_OUTPUT):
    # Output klasörünü bulmaya çalış
    output_root = "/content/output"
    candidates = sorted(glob.glob(os.path.join(output_root, "4DGS_*")))
    if candidates:
        LOCAL_OUTPUT = candidates[-1]
    else:
        print("❌ HATA: Model klasörü bulunamadı!")
        sys.exit(1)

model_path = LOCAL_OUTPUT
NET_WIDTH = 128 # Checkpoint boyutu

cmd_args = [
    sys.executable, "export_perframe_3DGS.py",
    "--source_path", source_path,
    "--model_path", model_path,
    "--iteration", str(ITERATIONS),
    "--skip_video",
    "--net_width", str(NET_WIDTH)
]

print(f"📂 Model: {model_path}")
print("⏳ İşleniyor (Bu sefer kareleri tek tek sayacak)...")

work_dir = "/content/4DGaussians-Enhanced"
try:
    process = subprocess.Popen(
        cmd_args,
        cwd=work_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True
    )

    # Logları izle
    for line in process.stdout:
        print(line, end="")

    process.wait()

except Exception as e:
    print(f"❌ Hata: {e}")

# --- 3. SONUÇLARI TOPLA VE DRIVE'A AT ---
# Scriptin içinde gördük: output_path = os.path.join(args.model_path,"gaussian_pertimestamp")
output_folder_name = "gaussian_pertimestamp"
ply_source_dir = os.path.join(model_path, output_folder_name)

if os.path.exists(ply_source_dir):
    ply_files = glob.glob(os.path.join(ply_source_dir, "*.ply"))
    count = len(ply_files)

    if count > 0:
        print("\n" + "="*60)
        print(f"✅ BAŞARILI! {count} adet PLY dosyası üretildi.")
        print(f"📁 Kaynak: {ply_source_dir}")

        # Drive'a Yedekle
        model_name = os.path.basename(model_path)
        drive_ply_dir = os.path.join("/content/drive/MyDrive/4DGS_project/output", model_name, "ply_sequence")

        print(f"\n📤 Drive'a Yedekleniyor: {drive_ply_dir}")
        if os.path.exists(drive_ply_dir):
            shutil.rmtree(drive_ply_dir)
        shutil.copytree(ply_source_dir, drive_ply_dir)

        print("🎉 İŞLEM TAMAMLANDI!")
        print("="*60)
    else:
        print("\n⚠️ Klasör var ama içi boş. Garip.")
else:
    print(f"\n❌ HATA: Beklenen '{output_folder_name}' klasörü oluşmadı.")

🚑 SCRIPT TAMİRİ VE EXPORT BAŞLATMA
🔧 Hedef Dosya: /content/4DGaussians-Enhanced/export_perframe_3DGS.py
   🐛 Hatalı döngü bulundu, düzeltiliyor...
   ✅ Yama başarıyla uygulandı!

🎬 EXPORT START
📂 Model: /content/output/4DGS_20251222_1530
⏳ İşleniyor (Bu sefer kareleri tek tek sayacak)...
  File "/content/4DGaussians-Enhanced/export_perframe_3DGS.py", line 102
    cameras = scene.getTestCameras()
IndentationError: unexpected indent

✅ BAŞARILI! 1 adet PLY dosyası üretildi.
📁 Kaynak: /content/output/4DGS_20251222_1530/gaussian_pertimestamp

📤 Drive'a Yedekleniyor: /content/drive/MyDrive/4DGS_project/output/4DGS_20251222_1530/ply_sequence
🎉 İŞLEM TAMAMLANDI!


In [ ]:
import sys
import subprocess
import os
import shutil

print("🚀 Manuel düzeltme sonrası Export (Final Deneme)...")

DATASET_PATH = "/content/data/my_scene/colmap_output"
output_root = "/content/output"
candidates = sorted([os.path.join(output_root, d) for d in os.listdir(output_root) if "4DGS_" in d])

if candidates:
    MODEL_PATH = candidates[-1]

    cmd_args = [
        sys.executable, "export_perframe_3DGS.py",
        "--source_path", DATASET_PATH,
        "--model_path", MODEL_PATH,
        "--iteration", "80000",
        "--skip_video",
        "--net_width", "128"
    ]

    print(f"📂 Model: {MODEL_PATH}")
    print("⏳ İşleniyor...")

    process = subprocess.Popen(
        cmd_args,
        cwd="/content/4DGaussians-Enhanced",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True
    )

    for line in process.stdout:
        print(line, end="")

    process.wait()

    # SONUÇ
    output_folder = os.path.join(MODEL_PATH, "gaussian_pertimestamp")
    if os.path.exists(output_folder):
        count = len(os.listdir(output_folder))
        print(f"\n✅ İŞLEM SONUCU: {count} dosya üretildi.")

        if count > 1:
            # Drive'a at
            model_name = os.path.basename(MODEL_PATH)
            drive_ply_dir = os.path.join("/content/drive/MyDrive/4DGS_project/output", model_name, "ply_sequence")
            if os.path.exists(drive_ply_dir): shutil.rmtree(drive_ply_dir)
            shutil.copytree(output_folder, drive_ply_dir)
            print(f"📤 Drive'a yedeklendi: {drive_ply_dir}")
    else:
        print("\n❌ Klasör oluşmadı.")
else:
    print("❌ Model bulunamadı.")

🚀 Manuel düzeltme sonrası Export (Final Deneme)...
📂 Model: /content/output/4DGS_20251222_1530
⏳ İşleniyor...
Looking for config file in /content/output/4DGS_20251222_1530/cfg_args
Config file found: /content/output/4DGS_20251222_1530/cfg_args
Rendering  /content/output/4DGS_20251222_1530
feature_dim: 128 [22/12 18:55:57]
Loading trained model at iteration 80000 [22/12 18:55:57]

Reading camera 1/8
Reading camera 2/8
Reading camera 3/8
Reading camera 4/8
Reading camera 5/8
Reading camera 6/8
Reading camera 7/8
Reading camera 8/8 [22/12 18:55:58]
Loading Training Cameras [22/12 18:55:58]
Loading Test Cameras [22/12 18:55:58]
Loading Video Cameras [22/12 18:55:58]
Deformation Net Set aabb [10.146437   2.8368807  7.1748652] [ -8.10082    -2.0269814 -11.853425 ] [22/12 18:55:58]
Voxel Plane: set aabb= Parameter containing:
tensor([[ 10.1464,   2.8369,   7.1749],
        [ -8.1008,  -2.0270, -11.8534]]) [22/12 18:55:58]
loading model from exists/content/output/4DGS_20251222_1530/point_cloud

In [ ]:
import os
import sys
import shutil
import subprocess

print("="*60)
print("🎬 CELL 13.6: 66 KARE EXPORTER (CONFIG FINDER)")
print("="*60)

# --- AYARLAR ---
TOTAL_FRAMES = 66
NET_WIDTH = 128
OUTPUT_ROOT = "/content/output"

# Model yolunu bul
candidates = sorted([os.path.join(OUTPUT_ROOT, d) for d in os.listdir(OUTPUT_ROOT) if "4DGS_" in d])
if not candidates:
    print("❌ Model bulunamadı!")
    sys.exit(1)
MODEL_PATH = candidates[-1]

# --- SCRİPT ---
script_content = f"""
import torch
import os
import sys
import numpy as np
from scene import Scene
from gaussian_renderer import GaussianModel
from utils.render_utils import get_state_at_time
from plyfile import PlyData, PlyElement

# --- YARDIMCI FONKSİYONLAR ---
def construct_list_of_attributes(feature_dc_shape, feature_rest_shape, scaling_shape, rotation_shape):
    l = ['x', 'y', 'z', 'nx', 'ny', 'nz']
    for i in range(feature_dc_shape[1]*feature_dc_shape[2]): l.append('f_dc_{{}}'.format(i))
    for i in range(feature_rest_shape[1]*feature_rest_shape[2]): l.append('f_rest_{{}}'.format(i))
    l.append('opacity')
    for i in range(scaling_shape[1]): l.append('scale_{{}}'.format(i))
    for i in range(rotation_shape[1]): l.append('rot_{{}}'.format(i))
    return l

def init_3DGaussians_ply(points, scales, rotations, opactiy, shs, feature_shape):
    xyz = points.detach().cpu().numpy()
    normals = np.zeros_like(xyz)
    feature_dc = shs[:,0:feature_shape[0],:]
    feature_rest = shs[:,feature_shape[0]:,:]
    f_dc = shs[:,:feature_shape[0],:].detach().transpose(1,2).flatten(start_dim=1).contiguous().cpu().numpy()
    f_rest = shs[:,feature_shape[0]:,:].detach().transpose(1,2).flatten(start_dim=1).contiguous().cpu().numpy()
    opacities = opactiy.detach().cpu().numpy()
    scale = scales.detach().cpu().numpy()
    rotation = rotations.detach().cpu().numpy()
    dtype_full = [(attribute, 'f4') for attribute in construct_list_of_attributes(feature_dc.shape, feature_rest.shape, scales.shape, rotations.shape)]
    elements = np.empty(xyz.shape[0], dtype=dtype_full)
    attributes = np.concatenate((xyz, normals, f_dc, f_rest, opacities, scale, rotation), axis=1)
    elements[:] = list(map(tuple, attributes))
    el = PlyElement.describe(elements, 'vertex')
    return PlyData([el])

def find_config_file():
    # Repoyu tara ve kplanes_config.py dosyasını bul
    repo_path = "/content/4DGaussians-Enhanced"
    for root, dirs, files in os.walk(repo_path):
        if "kplanes_config.py" in files:
            return os.path.join(root, "kplanes_config.py")
    return None

def export_66_frames():
    model_path = "{MODEL_PATH}"
    target_frames = {TOTAL_FRAMES}

    # 1. ARGS SİMÜLASYONU
    class MockArgs:
        def __init__(self):
            self.source_path = "/content/data/my_scene/colmap_output"
            self.model_path = model_path
            self.images = "images"
            self.eval = False; self.llffhold = 8; self.resolution = -1
            self.white_background = False; self.data_device = "cuda"
            self.sh_degree = 3;
            self.net_width = {NET_WIDTH}
            self.timebase_pe = 4; self.defor_depth = 1; self.posebase_pe = 10
            self.scale_rotation_pe = 2; self.opacity_pe = 2; self.timenet_width = 64
            self.timenet_output = 32; self.bounds = 1.6; self.plane_tv_weight = 0.0001
            self.time_smoothness_weight = 0.01; self.l1_time_planes = 0.0001
            # Config dosyasını dinamik yükleyeceğiz
            self.kplanes_config = None
            self.multires = [1, 2, 4, 8]
            self.no_dx = False; self.no_grid = False; self.no_ds = False
            self.no_dr = False; self.no_do = False; self.no_dshs = False
            self.empty_voxel = False; self.grid_pe = 0
            self.static_mlp = False; self.apply_rotation = False

    args = MockArgs()

    # 2. CONFIG DOSYASINI BUL VE YÜKLE
    real_config_path = find_config_file()
    if real_config_path:
        print(f"📄 Config bulundu: {{real_config_path}}")
        config_dict = {{}}
        with open(real_config_path, 'r') as f:
            exec(f.read(), {{}}, config_dict)
        if 'grid_config' in config_dict:
            args.kplanes_config = config_dict['grid_config']
            print("✅ Grid ayarları yüklendi.")
        else:
            raise ValueError("Config dosyasında 'grid_config' değişkeni yok!")
    else:
        raise FileNotFoundError("kplanes_config.py dosyası repo içinde bulunamadı!")

    # 3. MODELİ BAŞLAT
    print("🚀 Model yükleniyor...")
    gaussians = GaussianModel(args.sh_degree, args)
    scene = Scene(args, gaussians, load_iteration=80000, shuffle=False)

    # 4. KAMERA VE DÖNGÜ
    cameras = scene.getTrainCameras()
    if not cameras: cameras = scene.getTestCameras()
    if not cameras:
        print("❌ Hata: Kamera yok."); return

    ref_cam = cameras[0]

    output_dir = os.path.join(model_path, "ply_sequence_66")
    os.makedirs(output_dir, exist_ok=True)

    print(f"🎬 {{target_frames}} Kare işleniyor...")

    for i in range(target_frames):
        normalized_time = i / (target_frames - 1)
        ref_cam.timestamp = normalized_time

        points, scales, rots, opacity, shs = get_state_at_time(gaussians, ref_cam)

        feature_dc_shape = gaussians._features_dc.shape[1]
        feature_rest_shape = gaussians._features_rest.shape[1]
        gs_ply = init_3DGaussians_ply(points, scales, rots, opacity, shs, [feature_dc_shape, feature_rest_shape])

        filename = f"frame_{{i:05d}}.ply"
        gs_ply.write(os.path.join(output_dir, filename))

        if i % 10 == 0:
            print(f"   Frame {{i}}/{{target_frames}}")

    print(f"🎉 BİTTİ: {{output_dir}}")

if __name__ == "__main__":
    with torch.no_grad():
        export_66_frames()
"""

# Scripti yaz
script_path = "/content/4DGaussians-Enhanced/export_manual_66.py"
with open(script_path, "w") as f:
    f.write(script_content)

print("📝 Script hazırlandı. Çalıştırılıyor...")

# Çalıştır
cmd = [sys.executable, script_path]
process = subprocess.Popen(cmd, cwd="/content/4DGaussians-Enhanced", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    print(line, end="")
process.wait()

# Yedekle
output_local = os.path.join(MODEL_PATH, "ply_sequence_66")
if os.path.exists(output_local):
    count = len(os.listdir(output_local))
    print("\n" + "="*60)
    print(f"📊 FİNAL RAPOR: {count} adet PLY dosyası.")
    if count == TOTAL_FRAMES:
        model_name = os.path.basename(MODEL_PATH)
        drive_dest = os.path.join("/content/drive/MyDrive/4DGS_project/output", model_name, "ply_sequence_66")
        print(f"📤 Drive'a aktarılıyor...")
        if os.path.exists(drive_dest): shutil.rmtree(drive_dest)
        shutil.copytree(output_local, drive_dest)
        print("✅ YEDEKLEME TAMAMLANDI.")
    else:
        print(f"⚠️ Eksik dosya: {count}/{TOTAL_FRAMES}")
else:
    print("❌ HATA: Klasör oluşmadı.")

🎬 CELL 13.6: 66 KARE EXPORTER (CONFIG FINDER)
📝 Script hazırlandı. Çalıştırılıyor...
Traceback (most recent call last):
  File "/content/4DGaussians-Enhanced/export_manual_66.py", line 127, in <module>
    export_66_frames()
  File "/content/4DGaussians-Enhanced/export_manual_66.py", line 87, in export_66_frames
    raise FileNotFoundError("kplanes_config.py dosyası repo içinde bulunamadı!")
FileNotFoundError: kplanes_config.py dosyası repo içinde bulunamadı!
❌ HATA: Klasör oluşmadı.
